# PEMDT v2 — Complete Pipeline

**Probabilistic and Explainable Multimodal Digital Twin**  
MIMIC-IV ICU Edition · 64,691 patients · 9,890,935 observation windows

| Cell | Purpose | Key Outputs |
|---|---|---|
| 1 | Main Pipeline (Steps 0–7) | `PEMDT_dashboard_cohort.csv`, `PEMDT_cohort_summary.csv` |
| 2 | Build Patient Summary | `patient_summary.csv` |
| 3 | Raw HMM Validation | ROC/PR curves, group comparison, lead-time, subgroup AUC |
| 4 | Calibrated Validation (Platt Scaling) | `roc_pr_calibrated.png`, `calibration_plot.png`, `calibration_coefficients.png`, `raw_vs_calibrated_scatter.png` |
| 5 | Standalone LR Benchmark | Upper-bound discriminative reference (AUC=0.740) |
| 6 | Experiment Sweep (180 configurations) | `experiment_results.csv`, AUC heatmap, sensitivity/FAR plots |
| 7 | LLR vs δf Attribution Comparison | Feature agreement analysis across all windows |
| 8 | Explainability Visualizations | `explainability_plot1_frequency.png`, `explainability_plot2_difference.png`, `explainability_plot3_agreement.png` |

**Run order:** Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 6 → Cell 7 → Cell 8

**Key results:**
- Raw HMM AUC: 0.687 [0.683–0.692]
- Calibrated AUC: 0.742 · Brier: 0.185 · AUC gain: +0.057
- Vasopressor lead-time median: 720 min (12.0 hr)
- Death lead-time median: 4,500 min (3.13 days)
- LLR vs δf feature agreement: 19.0% — lab biomarkers nearly invisible to δf

**Required files (MIMIC-IV):**  
`hosp/` → `patients.csv`, `admissions.csv`, `labevents.csv`  
`icu/` → `icustays.csv`, `chartevents.csv`, `inputevents.csv`  
`data/labeled/` → CASAS smart-home CSVs (81 patients, architecture demo)

## Cell 1 — Main Pipeline

In [ ]:
# ============================================================
# PEMDT v2 — Multi-Patient Physiological-Behavioral Digital Twin
#             MIMIC-IV ICU Edition
# ============================================================
#
#  Architecture:
#    • Bayesian Hidden Markov Model  (Stable / Unstable)
#    • Age-adjusted prior belief and transition dynamics
#    • Glass-box explainability via Log-Likelihood Ratio (Λf)
#    • Multi-patient cohort pipeline
#    • CASAS behavioral fusion (architecture demonstration)
#
#  Directory layout expected:
#    ./data/mimiciv/icu/   → chartevents.csv, icustays.csv,
#                            inputevents.csv (vasopressors)
#    ./data/mimiciv/hosp/  → patients.csv, admissions.csv,
#                            labevents.csv, d_labitems.csv
#    ./labeled/            → CASAS smart-home CSVs
# ============================================================

import os
import gc
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm

warnings.filterwarnings("ignore")

# ============================================================
# GLOBAL PARAMETERS
# ============================================================

MIMIC_ICU_DIR     = "../data/mimiciv/icu"
MIMIC_HOSP_DIR    = "../data/mimiciv/hosp"
CASAS_LABELED_DIR = "../data/labeled"

WINDOW_MIN  = 5
ANCHOR_DATE = pd.Timestamp("2026-02-17 08:00:00")
CHUNK_SIZE  = 500_000
MAX_PATIENTS = None   # Set e.g. 500 for fast testing, None for full run

# ============================================================
# ITEMID CONSTANTS
# ============================================================

CHART_ITEMIDS = {
    "HR_bpm":    [220045],
    "SBP":       [220179, 220050, 224167, 227243],
    "DBP":       [220180, 220051, 224643, 227242],
    "SpO2":      [220277, 646],
    "Resp_rate": [220210, 224422],
    "Temp_C":    [223762, 226329],
    "MAP":       [220052, 220181, 225312],
    "GCS_total": [220739, 223900, 223901],
}

VALID_RANGE = {
    "HR_bpm":    (20,  300),
    "SBP":       (40,  260),
    "DBP":       (20,  200),
    "SpO2":      (50,  100),
    "Resp_rate": (4,   60),
    "Temp_C":    (30,  43),
    "MAP":       (20,  200),
    "GCS_total": (3,   15),
}

LAB_ITEMIDS = {
    "Lactate":    [50813],
    "WBC":        [51301, 51300],
    "Creatinine": [50912],
    "Sodium":     [50983],
    "Hemoglobin": [51222],
}

LAB_VALID_RANGE = {
    "Lactate":    (0.1, 30),
    "WBC":        (0.1, 200),
    "Creatinine": (0.1, 30),
    "Sodium":     (100, 180),
    "Hemoglobin": (1,   25),
}

VASOPRESSOR_ITEMIDS = [221906, 221289, 221749, 222315, 221662, 221653]

# ============================================================
# STEP 0 — PATIENT DISCOVERY
# ============================================================

def discover_patients(mimic_icu_dir, casas_dir):
    icustays_path = os.path.join(mimic_icu_dir, "icustays.csv")
    if not os.path.exists(icustays_path):
        raise FileNotFoundError(f"icustays.csv not found in {mimic_icu_dir}")

    icustays    = pd.read_csv(icustays_path, usecols=["subject_id", "stay_id"])
    all_subjects = sorted(icustays["subject_id"].unique().tolist())
    if MAX_PATIENTS:
        all_subjects = all_subjects[:MAX_PATIENTS]

    patient_ids = [f"p{sid}" for sid in all_subjects]

    casas_root  = Path(casas_dir)
    casas_files = sorted([f for f in casas_root.iterdir()
                          if f.is_file() and f.suffix.lower() == ".csv"])
    n_casas   = min(len(all_subjects), len(casas_files))
    casas_map = {f"p{all_subjects[i]}": str(casas_files[i])
                 for i in range(n_casas)}

    print(f"✅ Total ICU patients: {len(patient_ids)}")
    print(f"   CASAS-paired (architecture demo): {len(casas_map)}")
    return patient_ids, casas_map


# ============================================================
# STEP 1 — DEMOGRAPHICS + GROUND TRUTH
# ============================================================

def load_demographics(mimic_hosp_dir):
    patients   = pd.read_csv(os.path.join(mimic_hosp_dir, "patients.csv"))
    admissions = pd.read_csv(
        os.path.join(mimic_hosp_dir, "admissions.csv"),
        usecols=["subject_id", "hadm_id", "race", "insurance",
                 "discharge_location", "hospital_expire_flag",
                 "admittime", "dischtime"]
    )
    patients["dod"]        = pd.to_datetime(patients["dod"], errors="coerce")
    patients["is_deceased"] = patients["dod"].notna().astype(int)
    adm_first = (admissions.sort_values("admittime")
                            .drop_duplicates("subject_id"))
    demo = patients.merge(adm_first, on="subject_id", how="left")
    print(f"✅ Demographics loaded: {len(demo)} patients  "
          f"({demo['is_deceased'].sum()} deceased, "
          f"{demo['is_deceased'].mean()*100:.1f}%)")
    return demo


# ============================================================
# STEP 2 — CLINICAL ANCHOR EVENTS
# ============================================================

def load_anchor_events(mimic_icu_dir, subject_ids):
    subject_set = set(subject_ids)
    events = []

    inp_path = os.path.join(mimic_icu_dir, "inputevents.csv")
    if os.path.exists(inp_path):
        print("  Loading inputevents for vasopressor anchors...")
        for chunk in pd.read_csv(
            inp_path,
            usecols=["subject_id", "stay_id", "starttime", "itemid", "amount"],
            chunksize=CHUNK_SIZE, low_memory=False,
        ):
            mask = (chunk["subject_id"].isin(subject_set) &
                    chunk["itemid"].isin(VASOPRESSOR_ITEMIDS) &
                    chunk["amount"] > 0)
            events.append(chunk[mask])

    if events:
        vaso_df = pd.concat(events, ignore_index=True)
        vaso_df["starttime"] = pd.to_datetime(vaso_df["starttime"], errors="coerce")
        first_vaso = (vaso_df.groupby("subject_id")["starttime"]
                              .min().reset_index()
                              .rename(columns={"starttime": "first_vasopressor_time"}))
    else:
        first_vaso = pd.DataFrame(columns=["subject_id", "first_vasopressor_time"])

    icustays = pd.read_csv(
        os.path.join(mimic_icu_dir, "icustays.csv"),
        usecols=["subject_id", "stay_id", "intime", "outtime", "los"]
    )
    icustays["intime"]  = pd.to_datetime(icustays["intime"],  errors="coerce")
    icustays["outtime"] = pd.to_datetime(icustays["outtime"], errors="coerce")
    first_stay = (icustays.sort_values("intime")
                           .drop_duplicates("subject_id")
                           [["subject_id", "stay_id", "intime", "outtime", "los"]])

    anchor_df = first_stay.merge(first_vaso, on="subject_id", how="left")
    print(f"✅ Anchor events: {len(anchor_df)} patients, "
          f"{first_vaso['subject_id'].nunique()} with vasopressor data")
    return anchor_df


# ============================================================
# STEP 3 — CHARTEVENTS FEATURE EXTRACTION
# ============================================================

def load_chartevents(mimic_icu_dir, subject_ids):
    path = os.path.join(mimic_icu_dir, "chartevents.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"chartevents.csv not found in {mimic_icu_dir}")

    all_itemids = {iid for ids in CHART_ITEMIDS.values() for iid in ids}
    subject_set = set(subject_ids)
    chunks = []

    print("  Reading chartevents in chunks...")
    for i, chunk in enumerate(pd.read_csv(
        path,
        usecols=["subject_id", "stay_id", "charttime", "itemid", "valuenum"],
        chunksize=CHUNK_SIZE, low_memory=False,
    )):
        mask = (chunk["subject_id"].isin(subject_set) &
                chunk["itemid"].isin(all_itemids) &
                chunk["valuenum"].notna())
        filtered = chunk[mask]
        if not filtered.empty:
            chunks.append(filtered)
        if (i + 1) % 20 == 0:
            print(f"    ...chunk {i+1} processed")

    df = pd.concat(chunks, ignore_index=True)
    df["charttime"] = pd.to_datetime(df["charttime"], errors="coerce")
    df = df.dropna(subset=["charttime"])
    print(f"  Chartevents loaded: {len(df):,} rows")
    return df


def pivot_vitals(chartevents):
    itemid_to_feature = {}
    for feat, ids in CHART_ITEMIDS.items():
        for iid in ids:
            itemid_to_feature[iid] = feat

    ce = chartevents.copy()
    ce["feature"] = ce["itemid"].map(itemid_to_feature)
    ce = ce.dropna(subset=["feature"])

    valid_mask = pd.Series(True, index=ce.index)
    for feat, (lo, hi) in VALID_RANGE.items():
        m = ce["feature"] == feat
        valid_mask &= ~(m & ((ce["valuenum"] < lo) | (ce["valuenum"] > hi)))
    ce = ce[valid_mask]

    ce = ce.set_index("charttime").sort_index()
    ce["window"] = ce.index.floor("5min")

    agg = (
        ce.groupby(["subject_id", "stay_id", "window", "feature"])["valuenum"]
          .mean()
          .unstack("feature")
          .reset_index()
          .rename(columns={"window": "timestamp"})
    )

    hrv = (
        ce[ce["feature"] == "HR_bpm"]
          .groupby(["subject_id", "stay_id", "window"])["valuenum"]
          .std()
          .reset_index()
          .rename(columns={"window": "timestamp", "valuenum": "HRV_proxy"})
    )
    agg = agg.merge(hrv, on=["subject_id", "stay_id", "timestamp"], how="left")
    agg["HRV_proxy"] = agg["HRV_proxy"].fillna(0.05)
    agg["patient_id"] = "p" + agg["subject_id"].astype(str)

    print(f"✅ Vitals pivoted: {agg.shape}")
    return agg


# ============================================================
# STEP 4 — LABEVENTS ENRICHMENT
# ============================================================

def load_labevents(mimic_hosp_dir, subject_ids):
    path = os.path.join(mimic_hosp_dir, "labevents.csv")
    if not os.path.exists(path):
        print("  ⚠ labevents.csv not found — skipping lab enrichment.")
        return pd.DataFrame()

    all_itemids = {iid for ids in LAB_ITEMIDS.values() for iid in ids}
    subject_set = set(subject_ids)
    chunks = []

    print("  Reading labevents in chunks...")
    for chunk in pd.read_csv(
        path,
        usecols=["subject_id", "charttime", "itemid", "valuenum"],
        chunksize=CHUNK_SIZE, low_memory=False,
    ):
        mask = (chunk["subject_id"].isin(subject_set) &
                chunk["itemid"].isin(all_itemids) &
                chunk["valuenum"].notna())
        filtered = chunk[mask]
        if not filtered.empty:
            chunks.append(filtered)

    if not chunks:
        print("  ⚠ No lab events found.")
        return pd.DataFrame()

    df = pd.concat(chunks, ignore_index=True)
    df["charttime"] = pd.to_datetime(df["charttime"], errors="coerce")
    df = df.dropna(subset=["charttime"])

    itemid_to_feature = {}
    for feat, ids in LAB_ITEMIDS.items():
        for iid in ids:
            itemid_to_feature[iid] = feat
    df["feature"] = df["itemid"].map(itemid_to_feature)

    valid_mask = pd.Series(True, index=df.index)
    for feat, (lo, hi) in LAB_VALID_RANGE.items():
        m = df["feature"] == feat
        valid_mask &= ~(m & ((df["valuenum"] < lo) | (df["valuenum"] > hi)))
    df = df[valid_mask]

    df["window"] = df["charttime"].dt.floor("5min")
    labs_pivot = (
        df.groupby(["subject_id", "window", "feature"])["valuenum"]
          .mean()
          .unstack("feature")
          .reset_index()
          .rename(columns={"window": "timestamp"})
    )
    labs_pivot["patient_id"] = "p" + labs_pivot["subject_id"].astype(str)
    print(f"✅ Lab events pivoted: {labs_pivot.shape}")
    return labs_pivot


def merge_labs_into_vitals(vitals_df, labs_df):
    if labs_df.empty:
        for col in LAB_ITEMIDS.keys():
            vitals_df[col] = np.nan
        return vitals_df

    merged = vitals_df.merge(
        labs_df.drop(columns=["subject_id"], errors="ignore"),
        on=["patient_id", "timestamp"],
        how="left"
    )
    lab_cols = [c for c in LAB_ITEMIDS.keys() if c in merged.columns]
    merged = merged.sort_values(["patient_id", "timestamp"])
    merged[lab_cols] = (merged.groupby("patient_id")[lab_cols]
                               .transform(lambda x: x.ffill()))
    print(f"✅ Labs merged. Final shape: {merged.shape}")
    return merged


# ============================================================
# STEP 5 — CASAS BEHAVIOURAL PROCESSING  (architecture demo)
# ============================================================

def most_common_activity(x):
    x = x.dropna().replace(["None", "nan", ""], np.nan).dropna()
    return x.value_counts().idxmax() if len(x) > 0 else "Unknown"


def process_casas_file(filepath, patient_id):
    try:
        df = pd.read_csv(
            filepath, header=None, engine="python", on_bad_lines="skip",
            names=["date", "time", "sensor", "status", "label"],
        )
        df["timestamp"] = pd.to_datetime(
            df["date"].astype(str) + " " + df["time"].astype(str),
            errors="coerce"
        )
        df = df.dropna(subset=["timestamp"]).sort_values("timestamp")
        df["timestamp"] = ANCHOR_DATE + (df["timestamp"] - df["timestamp"].min())
        df["label"]    = df["label"].fillna("Unknown").astype(str)
        df["activity"] = df["label"].str.extract(r"(.+?)=").fillna("Unknown").ffill()
        df["patient_id"] = patient_id
        return df[["timestamp", "sensor", "activity", "patient_id"]]
    except Exception as e:
        print(f"  [{patient_id}] CASAS error: {e}")
        return None


def load_casas_all_patients(patient_ids, casas_map):
    frames = []
    for pid in patient_ids:
        fp = casas_map.get(pid)
        if not fp:
            continue
        df = process_casas_file(fp, pid)
        if df is not None:
            frames.append(df)

    if not frames:
        print("  ⚠ No CASAS files loaded.")
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    print(f"✅ CASAS loaded: {combined.shape}  "
          f"(architecture demo — {len(frames)} patients)")
    return combined


# ============================================================
# STEP 6 — MULTIMODAL FUSION
# ============================================================

def fuse_all_patients(mimic_df, casas_df):
    """
    Merges CASAS behavioral context into MIMIC windows for the 81 paired
    patients. All other patients remain vitals-only with activity_state=Unknown.
    CASAS timestamps are re-anchored to each patient's MIMIC intime.
    """
    mimic_df = mimic_df.copy()
    mimic_df["activity_state"]      = "Unknown"
    mimic_df["activity_intensity"]  = 0.0
    mimic_df["activity"]            = "Unknown"
    mimic_df["has_behavioral_data"] = False

    if casas_df.empty:
        print("⚠ No CASAS data — returning vitals-only cohort.")
        return mimic_df

    casas_pids       = casas_df["patient_id"].unique()
    casas_agg_frames = []

    for pid in casas_pids:
        c = casas_df[casas_df["patient_id"] == pid].copy()
        c["activity"] = c["activity"].fillna("Unknown")

        res = (
            c.set_index("timestamp")
             .resample("5min")
             .agg({"sensor": "count", "activity": most_common_activity})
             .rename(columns={"sensor": "activity_intensity_casas"})
             .reset_index()
        )
        res["activity_casas"] = res["activity"]
        res["patient_id"]     = pid

        patient_mimic = mimic_df[mimic_df["patient_id"] == pid]["timestamp"]
        if patient_mimic.empty:
            continue

        offset          = patient_mimic.min() - res["timestamp"].min()
        res["timestamp"] = res["timestamp"] + offset
        casas_agg_frames.append(res)

    if not casas_agg_frames:
        print("⚠ No CASAS patients matched MIMIC data — returning vitals-only.")
        return mimic_df

    casas_agg = pd.concat(casas_agg_frames, ignore_index=True)

    mimic_casas     = mimic_df[mimic_df["patient_id"].isin(casas_pids)].copy()
    mimic_non_casas = mimic_df[~mimic_df["patient_id"].isin(casas_pids)].copy()

    print(f"  CASAS patients in MIMIC: {mimic_casas['patient_id'].nunique()}")
    print(f"  Non-CASAS patients:      {mimic_non_casas['patient_id'].nunique()}")

    mimic_casas = mimic_casas.sort_values("timestamp").reset_index(drop=True)
    casas_agg   = casas_agg.sort_values("timestamp").reset_index(drop=True)

    enriched_casas = pd.merge_asof(
        mimic_casas,
        casas_agg[["patient_id", "timestamp",
                   "activity_intensity_casas", "activity_casas"]],
        on="timestamp", by="patient_id",
        tolerance=pd.Timedelta("5min"), direction="nearest",
    )

    has_casas = enriched_casas["activity_intensity_casas"].notna()
    enriched_casas.loc[has_casas, "has_behavioral_data"] = True
    enriched_casas.loc[has_casas, "activity_intensity"]  = (
        enriched_casas.loc[has_casas, "activity_intensity_casas"])
    enriched_casas.loc[has_casas, "activity"] = (
        enriched_casas.loc[has_casas, "activity_casas"])
    enriched_casas = enriched_casas.drop(
        columns=["activity_intensity_casas", "activity_casas"], errors="ignore")

    for pid in enriched_casas[enriched_casas["has_behavioral_data"]]["patient_id"].unique():
        mask = enriched_casas["patient_id"] == pid
        med  = enriched_casas.loc[mask, "activity_intensity"].median()
        enriched_casas.loc[mask, "activity_state"] = np.where(
            enriched_casas.loc[mask, "activity_intensity"] > med,
            "Active", "Inactive"
        )

    enriched = pd.concat([enriched_casas, mimic_non_casas], ignore_index=True)
    enriched = enriched.sort_values(["patient_id", "timestamp"]).reset_index(drop=True)

    n_casas_patients = enriched[enriched["has_behavioral_data"]]["patient_id"].nunique()
    n_casas_windows  = enriched["has_behavioral_data"].sum()
    print(f"✅ Fusion complete: {enriched.shape}")
    print(f"   CASAS patients: {n_casas_patients}  |  CASAS windows: {n_casas_windows:,}")
    return enriched


# ============================================================
# STEP 7 — BAYESIAN HEALTH STATE ESTIMATION
# ============================================================

GLOBAL_STATS = {
    "SBP":        (120, 20),
    "HRV_proxy":  (0.15, 0.10),
    "SpO2":       (97,   3),
    "Resp_rate":  (18,   5),
    "MAP":        (80,   15),
    "Lactate":    (2.0,  1.5),
    "WBC":        (10,   5),
    "Creatinine": (1.2,  0.8),
}

FEATURE_WEIGHTS = {
    "SBP":        1.3,
    "HRV_proxy":  1.5,
    "SpO2":       1.4,
    "Resp_rate":  1.2,
    "MAP":        1.4,
    "Lactate":    1.6,
    "WBC":        1.0,
    "Creatinine": 1.1,
    "activity":   0.8,
}

UNSTABLE_Z = {
    "SBP":        -2.0,
    "HRV_proxy":  -2.0,
    "SpO2":       -2.0,
    "Resp_rate":  +2.0,
    "MAP":        -2.0,
    "Lactate":    +2.0,
    "WBC":        +2.0,
    "Creatinine": +2.0,
}


def bayesian_health_state_patient(df):
    """
    Forward-filtering Bayesian HMM with:
      - Age-adjusted prior belief (4 brackets)
      - Age-adjusted transition matrix
      - Clinical feature weighting
      - Global z-score normalisation (population-level)
      - Adaptive momentum: higher persistence once instability detected
      - Log-sum-exp normalisation for numerical stability
      - Behavioral context from CASAS where available
    """
    df = df.sort_values("timestamp").reset_index(drop=True)

    age = df["anchor_age"].iloc[0] if "anchor_age" in df.columns else 65
    age = age if pd.notna(age) and 18 < age < 110 else 65

    if age < 45:
        belief_t   = np.array([0.97, 0.03])
        base_trans = np.array([[0.96, 0.04], [0.20, 0.80]])
    elif age < 65:
        belief_t   = np.array([0.95, 0.05])
        base_trans = np.array([[0.95, 0.05], [0.15, 0.85]])
    elif age < 80:
        belief_t   = np.array([0.92, 0.08])
        base_trans = np.array([[0.93, 0.07], [0.10, 0.90]])
    else:
        belief_t   = np.array([0.88, 0.12])
        base_trans = np.array([[0.90, 0.10], [0.08, 0.92]])

    unstable_probs = []

    for _, row in df.iterrows():
        transition_matrix = base_trans.copy()
        if belief_t[1] > 0.4:
            transition_matrix[1, 1] = 0.95
            transition_matrix[1, 0] = 0.05

        belief_pred = belief_t @ transition_matrix
        log_l_s = 0.0
        log_l_u = 0.0

        for feat in ["SBP", "HRV_proxy", "SpO2", "Resp_rate",
                     "MAP", "Lactate", "WBC", "Creatinine"]:
            val = row.get(feat, np.nan)
            if pd.notna(val):
                mu, sd     = GLOBAL_STATS[feat]
                z          = (val - mu) / sd
                w          = FEATURE_WEIGHTS[feat]
                z_unstable = UNSTABLE_Z[feat]
                log_l_s   += w * norm.logpdf(z, loc=0,          scale=1.0)
                log_l_u   += w * norm.logpdf(z, loc=z_unstable, scale=1.5)

        has_beh = row.get("has_behavioral_data", False)
        act     = row.get("activity_state", "Unknown")
        if has_beh and act in ["Active", "Cooking", "Walking"]:
            log_l_s += FEATURE_WEIGHTS["activity"] * np.log(0.7)
            log_l_u += FEATURE_WEIGHTS["activity"] * np.log(0.3)
        elif has_beh:
            log_l_s += FEATURE_WEIGHTS["activity"] * np.log(0.9)
            log_l_u += FEATURE_WEIGHTS["activity"] * np.log(0.1)

        log_post_s = np.log(belief_pred[0] + 1e-300) + log_l_s
        log_post_u = np.log(belief_pred[1] + 1e-300) + log_l_u
        log_max    = max(log_post_s, log_post_u)
        post_s     = np.exp(log_post_s - log_max)
        post_u     = np.exp(log_post_u - log_max)
        belief_t   = np.array([post_s, post_u]) / (post_s + post_u + 1e-9)
        unstable_probs.append(belief_t[1])

    df["unstable_probability"] = unstable_probs
    df["alert_triggered"]      = df["unstable_probability"] > 0.5
    return df


def run_bayesian_all_patients(cohort_df):
    frames = []
    pids   = cohort_df["patient_id"].unique()
    for i, pid in enumerate(pids):
        group  = cohort_df[cohort_df["patient_id"] == pid].copy()
        result = bayesian_health_state_patient(group)
        frames.append(result)
        if (i + 1) % 1000 == 0:
            print(f"  [{i+1}/{len(pids)}] processed...")
    final = pd.concat(frames, ignore_index=True)
    print(f"✅ Bayesian estimation complete. "
          f"Cohort alert rate: {final['alert_triggered'].mean():.3f}")
    return final


# ============================================================
# STEP 8 — GLASS-BOX EXPLAINABILITY (Log-Likelihood Ratio)
# ============================================================
#
#  Λf(t) = log P(of,t | s2) - log P(of,t | s1)
#
#  The feature with the largest |Λf(t)| is identified as the
#  primary driver. This correctly attributes risk to features
#  that currently resemble pathological values — not to
#  superficial signal variance or features outside the model.
#
#  Validated against change-based attribution (δf):
#  δf attributed risk to Heart Rate (>1M windows) which is
#  NOT in FEATURE_WEIGHTS and never enters the belief update.
#  LLR consistently identifies SpO2, Resp Rate, MAP, HRV —
#  all features that directly drive the Bayesian inference.

FEATURE_LABELS = {
    "SBP":        "Systolic BP",
    "HRV_proxy":  "HRV (variability)",
    "SpO2":       "Oxygen Saturation",
    "Resp_rate":  "Respiratory Rate",
    "MAP":        "Mean Arterial Pressure",
    "Lactate":    "Lactate",
    "WBC":        "White Cell Count",
    "Creatinine": "Creatinine",
}


def generate_explanations(df):
    """
    Generate per-window clinical explanations using Log-Likelihood Ratio.
    For each window, identifies the feature whose current value most
    strongly resembles an unstable patient relative to a stable one.
    """
    explanations = []
    df = df.reset_index(drop=True)

    for i in range(len(df)):
        if i == 0:
            explanations.append("Initial Baseline")
            continue

        row        = df.loc[i]
        delta_prob = (df.loc[i,   "unstable_probability"] -
                      df.loc[i-1, "unstable_probability"])

        best_feat = None
        best_llr  = -np.inf

        for feat in FEATURE_WEIGHTS:
            if feat == "activity":
                continue
            val = row.get(feat, np.nan)
            if pd.isna(val):
                continue
            mu, sd  = GLOBAL_STATS[feat]
            z       = (val - mu) / sd
            z_u     = UNSTABLE_Z[feat]
            w       = FEATURE_WEIGHTS[feat]
            log_l_s = w * (-0.5 * z**2)
            log_l_u = w * (-0.5 * ((z - z_u) / 1.5)**2)
            llr     = abs(log_l_u - log_l_s)
            if llr > best_llr:
                best_llr  = llr
                best_feat = feat

        key_label  = FEATURE_LABELS.get(best_feat, best_feat) if best_feat else "Unknown"
        act        = row.get("activity_state", "Unknown")
        has_beh    = row.get("has_behavioral_data", False)

        act_suffix = ""
        if has_beh and act not in ["Unknown", "Inactive"]:
            act_suffix = f" during {act} state"

        if delta_prob > 0.05:
            explanations.append(f"Risk Increase: {key_label}{act_suffix}")
        elif delta_prob < -0.05:
            explanations.append(f"Risk Decrease: Stabilised by {key_label}{act_suffix}")
        else:
            explanations.append("Stable")

    return explanations


def add_explanations_all_patients(cohort_df):
    frames = []
    for pid, group in cohort_df.groupby("patient_id"):
        group = group.copy().reset_index(drop=True)
        group["clinical_explanation"] = generate_explanations(group)
        frames.append(group)
    final = pd.concat(frames, ignore_index=True)
    print("✅ Explainability layer complete (LLR).")
    return final


# ============================================================
# MAIN PIPELINE
# ============================================================

def run_pipeline():
    print("\n========== PEMDT v2 Pipeline ==========\n")

    # Step 0: Discover patients
    patient_ids, casas_map = discover_patients(MIMIC_ICU_DIR, CASAS_LABELED_DIR)
    subject_ids = [int(pid.replace("p", "")) for pid in patient_ids]

    # Step 1: Demographics + anchor events
    print("\n--- Step 1: Demographics & Anchor Events ---")
    demo_df   = load_demographics(MIMIC_HOSP_DIR)
    anchor_df = load_anchor_events(MIMIC_ICU_DIR, subject_ids)

    # Step 2: Chartevents vitals
    print("\n--- Step 2: Vital Sign Extraction ---")
    chart_df  = load_chartevents(MIMIC_ICU_DIR, subject_ids)
    vitals_df = pivot_vitals(chart_df)
    del chart_df; gc.collect()
    vitals_df.to_csv("PEMDT_mimic_features.csv", index=False)

    # Step 3: Lab enrichment
    print("\n--- Step 3: Lab Enrichment ---")
    labs_df  = load_labevents(MIMIC_HOSP_DIR, subject_ids)
    enriched = merge_labs_into_vitals(vitals_df, labs_df)
    del labs_df; gc.collect()

    # Step 4: CASAS behavioural (architecture demo)
    print("\n--- Step 4: CASAS Behavioural Processing ---")
    casas_df = load_casas_all_patients(patient_ids, casas_map)
    if not casas_df.empty:
        casas_df.to_csv("PEMDT_casas_features.csv", index=False)

    # Step 5: Multimodal fusion
    print("\n--- Step 5: Multimodal Fusion ---")
    fused_df = fuse_all_patients(enriched, casas_df)

    # Join demographics + anchor events
    fused_df["subject_id"] = fused_df["patient_id"].str.replace("p", "").astype(int)
    for col in ["is_deceased", "dod", "anchor_age", "gender", "race",
                "insurance", "discharge_location", "hospital_expire_flag"]:
        if col in fused_df.columns:
            fused_df = fused_df.drop(columns=[col])
    fused_df = fused_df.merge(
        demo_df[["subject_id", "is_deceased", "dod", "anchor_age",
                 "gender", "race", "discharge_location", "hospital_expire_flag"]],
        on="subject_id", how="left"
    )
    fused_df = fused_df.merge(
        anchor_df[["subject_id", "intime", "outtime", "los",
                   "first_vasopressor_time"]],
        on="subject_id", how="left"
    )
    fused_df["is_deceased"] = fused_df["is_deceased"].fillna(0).astype(int)

    # Temporal censoring — remove post-mortem windows
    fused_df["timestamp"] = pd.to_datetime(fused_df["timestamp"])
    fused_df["dod"]       = pd.to_datetime(fused_df["dod"])
    fused_df = fused_df[
        (fused_df["dod"].isna()) | (fused_df["timestamp"] <= fused_df["dod"])
    ].copy()
    print(f"  Temporal censoring: {len(fused_df):,} windows remain across "
          f"{fused_df['patient_id'].nunique():,} patients")

    fused_df.to_csv("PEMDT_fused_cohort.csv", index=False)

    # Step 6: Bayesian estimation
    print("\n--- Step 6: Bayesian Health State Estimation ---")
    bayes_df = run_bayesian_all_patients(fused_df)
    del fused_df; gc.collect()
    bayes_df.to_csv("PEMDT_bayes_cohort.csv", index=False)

    # Step 7: Glass-box explainability (LLR)
    print("\n--- Step 7: Glass-Box Explainability (LLR) ---")
    final_df = add_explanations_all_patients(bayes_df)
    del bayes_df; gc.collect()
    final_df.to_csv("PEMDT_dashboard_cohort.csv", index=False)

    # Cohort summary
    print("\n--- Cohort Summary ---")
    agg_spec = {
        "alert_rate":       ("alert_triggered",      "mean"),
        "peak_instability": ("unstable_probability", "max"),
        "mean_instability": ("unstable_probability", "mean"),
        "std_instability":  ("unstable_probability", "std"),
        "n_windows":        ("unstable_probability", "count"),
    }
    for col in ["anchor_age", "gender", "is_deceased", "los",
                "discharge_location", "hospital_expire_flag"]:
        if col in final_df.columns:
            agg_spec[col] = (col, "first")
    summary = final_df.groupby("patient_id").agg(**agg_spec).round(3)
    summary.to_csv("PEMDT_cohort_summary.csv")
    print(summary.describe().to_string())

    print("\n✅ Pipeline complete. Output files:")
    for f in ["PEMDT_mimic_features.csv", "PEMDT_casas_features.csv",
              "PEMDT_fused_cohort.csv",   "PEMDT_bayes_cohort.csv",
              "PEMDT_dashboard_cohort.csv", "PEMDT_cohort_summary.csv"]:
        print(f"   {f}")


if __name__ == "__main__":
    run_pipeline()



========== PEMDT v2 Pipeline ==========

✅ Total ICU patients: 65366
   CASAS-paired (architecture demo): 81

--- Step 1: Demographics & Anchor Events ---
✅ Demographics loaded: 364627 patients  (38301 deceased, 10.5%)
  Loading inputevents for vasopressor anchors...
✅ Anchor events: 65366 patients, 23024 with vasopressor data

--- Step 2: Vital Sign Extraction ---
  Reading chartevents in chunks...
    ...chunk 20 processed
    ...chunk 40 processed
    ...chunk 60 processed
    ...chunk 80 processed
    ...chunk 100 processed
    ...chunk 120 processed
    ...chunk 140 processed
    ...chunk 160 processed
    ...chunk 180 processed
    ...chunk 200 processed
    ...chunk 220 processed
    ...chunk 240 processed
    ...chunk 260 processed
    ...chunk 280 processed
    ...chunk 300 processed
    ...chunk 320 processed
    ...chunk 340 processed
    ...chunk 360 processed
    ...chunk 380 processed
    ...chunk 400 processed
    ...chunk 420 processed
    ...chunk 440 processed
    ..

## Cell 2 — Build Patient Summary

In [2]:
# ============================================================
# Build patient_summary.csv
# Extends PEMDT_cohort_summary.csv with std_instability and
# discharge columns — required for standalone LR benchmark.
# Run AFTER run_pipeline() has completed.
# ============================================================

import pandas as pd

summary   = pd.read_csv("PEMDT_cohort_summary.csv")
summary["subject_id"] = summary["patient_id"].str.replace("p", "").astype(int)

# Add std_instability from dashboard
dashboard = pd.read_csv("PEMDT_dashboard_cohort.csv",
                         usecols=["patient_id", "unstable_probability"],
                         low_memory=False)
std_df = (dashboard.groupby("patient_id")["unstable_probability"]
                   .std().reset_index()
                   .rename(columns={"unstable_probability": "std_instability"}))

if "std_instability" not in summary.columns:
    summary = summary.merge(std_df, on="patient_id", how="left")

# Add discharge_location and hospital_expire_flag if not present
if "discharge_location" not in summary.columns:
    adm = pd.read_csv("../data/mimiciv/hosp/admissions.csv",
                       usecols=["subject_id", "discharge_location",
                                "hospital_expire_flag"])
    adm = adm.drop_duplicates("subject_id")
    summary = summary.merge(adm, on="subject_id", how="left")

summary.to_csv("patient_summary.csv", index=False)
print(f"✅ Saved patient_summary.csv — {len(summary)} patients")
print(f"   Columns: {summary.columns.tolist()}")


✅ Saved patient_summary.csv — 64691 patients
   Columns: ['patient_id', 'alert_rate', 'peak_instability', 'mean_instability', 'std_instability', 'n_windows', 'anchor_age', 'gender', 'is_deceased', 'los', 'discharge_location', 'hospital_expire_flag', 'subject_id']


## Cell 3 — Raw HMM Validation

In [3]:
# ============================================================
# PEMDT v2 — Raw HMM Validation  (Journal-Ready)
#
#  Metrics:
#    • ROC-AUC with 95% CI  (bootstrap, n=1000)
#    • PR-AUC with 95% CI
#    • Brier score
#    • Welch t-test + Mann-Whitney U  (alive vs deceased)
#    • Calibration plot
#    • Discharge destination analysis
#    • Lead-time against real clinical events (vasopressor, death)
#    • Instability trajectory plots
#    • Demographic subgroup AUC
#    • Subgroup AUC bar chart
#    • Violin group separation plot
#    • Overlapping density group separation plot
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy.stats import ttest_ind, mannwhitneyu, gaussian_kde
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

# ── CONFIG ────────────────────────────────────────────────────────────────
DASHBOARD_FILE  = "PEMDT_dashboard_cohort.csv"
PATIENTS_FILE   = "../../data/mimiciv/hosp/patients.csv"
ADMISSIONS_FILE = "../../data/mimiciv/hosp/admissions.csv"
OUTPUT_DIR      = Path("validation_output")
OUTPUT_DIR.mkdir(exist_ok=True)

ALERT_THRESHOLD = 0.3
BOOTSTRAP_N     = 1000
RANDOM_SEED     = 42

COLOR_ALIVE    = "#4C72B0"
COLOR_DECEASED = "#C44E52"

# ── 1. LOAD DATA ──────────────────────────────────────────────────────────

def load_data():
    dashboard = pd.read_csv(DASHBOARD_FILE, low_memory=False)
    dashboard["timestamp"] = pd.to_datetime(dashboard["timestamp"], errors="coerce")

    if "subject_id" not in dashboard.columns:
        dashboard["subject_id"] = (
            dashboard["patient_id"].str.replace("p", "", regex=False).astype(int))
    else:
        dashboard["subject_id"] = dashboard["subject_id"].astype(int)

    patients = pd.read_csv(PATIENTS_FILE)
    patients["dod"]         = pd.to_datetime(patients["dod"], errors="coerce")
    patients["is_deceased"] = patients["dod"].notna().astype(int)

    merge_cols = ["subject_id", "is_deceased", "dod"]
    for opt in ["anchor_age", "gender"]:
        if opt in patients.columns:
            merge_cols.append(opt)
    for col in [c for c in merge_cols if c != "subject_id"]:
        if col in dashboard.columns:
            dashboard = dashboard.drop(columns=[col])
    dashboard = dashboard.merge(patients[merge_cols], on="subject_id", how="left")

    if Path(ADMISSIONS_FILE).exists():
        adm = pd.read_csv(ADMISSIONS_FILE,
                          usecols=["subject_id", "discharge_location",
                                   "hospital_expire_flag"]).drop_duplicates("subject_id")
        for col in ["discharge_location", "hospital_expire_flag"]:
            if col in dashboard.columns:
                dashboard = dashboard.drop(columns=[col])
        dashboard = dashboard.merge(adm, on="subject_id", how="left")

    dashboard["is_deceased"]     = dashboard["is_deceased"].fillna(0).astype(int)
    dashboard["alert_triggered"] = dashboard["alert_triggered"].astype(bool)
    for tcol in ["first_vasopressor_time", "intime", "outtime"]:
        if tcol in dashboard.columns:
            dashboard[tcol] = pd.to_datetime(dashboard[tcol], errors="coerce")

    print(f"Dashboard shape: {dashboard.shape}")
    print(f"Columns: {dashboard.columns.tolist()}\n")
    print("Mortality distribution (patient level):")
    print(dashboard.groupby("patient_id")["is_deceased"].first().value_counts())
    return dashboard


# ── 2. PATIENT-LEVEL AGGREGATION ──────────────────────────────────────────

def aggregate_to_patient(dashboard):
    agg_spec = {
        "peak_instability": ("unstable_probability", "max"),
        "mean_instability": ("unstable_probability", "mean"),
        "std_instability":  ("unstable_probability", "std"),
        "n_windows":        ("unstable_probability", "count"),
        "is_deceased":      ("is_deceased",           "max"),
    }
    for col in ["subject_id", "anchor_age", "gender"]:
        if col in dashboard.columns:
            agg_spec[col] = (col, "first")
    if "alert_triggered" in dashboard.columns:
        agg_spec["alert_rate"] = ("alert_triggered", "mean")
    if "los" in dashboard.columns:
        agg_spec["los"] = ("los", "first")
    for col in ["discharge_location", "hospital_expire_flag"]:
        if col in dashboard.columns:
            agg_spec[col] = (col, "first")

    agg = dashboard.groupby("patient_id").agg(**agg_spec).reset_index()
    agg["is_deceased"] = agg["is_deceased"].fillna(0).astype(int)

    print(f"\nPatient-level summary — {len(agg)} patients, "
          f"{int(agg['is_deceased'].sum())} deaths "
          f"({agg['is_deceased'].mean()*100:.1f}%)")
    return agg


# ── 3. BOOTSTRAP CI ───────────────────────────────────────────────────────

def bootstrap_auc(y_true, y_score, metric_fn, n=BOOTSTRAP_N, seed=RANDOM_SEED):
    rng   = np.random.default_rng(seed)
    point = metric_fn(y_true, y_score)
    vals  = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            vals.append(metric_fn(y_true[idx], y_score[idx]))
        except Exception:
            continue
    lo, hi = np.percentile(vals, [2.5, 97.5]) if vals else (np.nan, np.nan)
    return point, lo, hi


# ── 4. ROC + PR CURVES ────────────────────────────────────────────────────

def plot_roc_pr(patient_summary):
    y_true  = patient_summary["is_deceased"].values
    y_score = patient_summary["mean_instability"].values

    roc_pt, roc_lo, roc_hi = bootstrap_auc(y_true, y_score, roc_auc_score)
    pr_pt,  pr_lo,  pr_hi  = bootstrap_auc(y_true, y_score, average_precision_score)
    brier = brier_score_loss(y_true, y_score)

    fpr, tpr, _   = roc_curve(y_true, y_score)
    prec, rec, _  = precision_recall_curve(y_true, y_score)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].plot(fpr, tpr, lw=2, color="steelblue", linestyle="-", marker="o", markevery=max(1, len(fpr)//10), markersize=5,
                 label=f"PEMDT  AUC = {roc_pt:.3f} [{roc_lo:.3f}-{roc_hi:.3f}]")
    axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Chance")
    axes[0].fill_between(fpr, tpr, alpha=0.1, color="steelblue")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("ROC Curve")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(rec, prec, lw=2, color="darkorange", linestyle="--", marker="s", markevery=max(1, len(rec)//10), markersize=5,
                 label=f"PEMDT  AP = {pr_pt:.3f} [{pr_lo:.3f}-{pr_hi:.3f}]")
    axes[1].axhline(y_true.mean(), color="k", linestyle="--", lw=1,
                    label=f"Baseline = {y_true.mean():.3f}")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision-Recall Curve")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "roc_pr_curves.png", dpi=150, bbox_inches="tight")
    plt.close()

    print(f"\nROC-AUC : {roc_pt:.3f}  95% CI [{roc_lo:.3f}-{roc_hi:.3f}]")
    print(f"PR-AUC  : {pr_pt:.3f}  95% CI [{pr_lo:.3f}-{pr_hi:.3f}]")
    print(f"Brier   : {brier:.3f}")

    return {"roc_auc": roc_pt, "roc_ci_lo": roc_lo, "roc_ci_hi": roc_hi,
            "pr_auc": pr_pt,   "pr_ci_lo": pr_lo,   "pr_ci_hi": pr_hi,
            "brier": brier}


# ── 5. GROUP COMPARISON (boxplot) ─────────────────────────────────────────

def group_comparison(patient_summary):
    dead  = patient_summary[patient_summary["is_deceased"] == 1]
    alive = patient_summary[patient_summary["is_deceased"] == 0]
    metrics = ["peak_instability", "mean_instability",
               "std_instability",  "alert_rate"]
    results = {}
    fig, axes = plt.subplots(1, len(metrics), figsize=(16, 5))
    for ax, metric in zip(axes, metrics):
        d_vals = dead[metric].dropna().values
        a_vals = alive[metric].dropna().values
        _, p_t = ttest_ind(d_vals, a_vals, equal_var=False)
        _, p_u = mannwhitneyu(d_vals, a_vals, alternative="two-sided")
        sig = "**" if p_t < 0.01 else ("*" if p_t < 0.05 else "ns")
        bp = ax.boxplot([a_vals, d_vals],
                   tick_labels=["Alive", "Deceased"],
                   patch_artist=True,
                   boxprops=dict(facecolor="lightblue", alpha=0.7),
                   medianprops=dict(color="navy", lw=2))
        for patch, hatch in zip(bp["boxes"], ["//", "xx"]):
            patch.set_hatch(hatch)
        ax.set_title(f"{metric}\np(t)={p_t:.4f}  {sig}", fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        results[metric] = {
            "dead_mean":  round(d_vals.mean(), 4),
            "alive_mean": round(a_vals.mean(), 4),
            "t_p":        round(p_t, 4),
            "mwu_p":      round(p_u, 4),
        }
        print(f"{metric:25s}  dead={d_vals.mean():.3f}  alive={a_vals.mean():.3f}  "
              f"p(t)={p_t:.4f}  p(U)={p_u:.4f}  {sig}")
    plt.suptitle("Instability Metrics: Alive vs Deceased")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "group_comparison.png", dpi=150)
    plt.close()
    return results


# ── 5b. GROUP SEPARATION VIOLIN PLOT ──────────────────────────────────────

def plot_violin_group(patient_summary):
    """Violin plots — shows full distribution shape per group."""
    metrics = [
        ("peak_instability", "Peak Instability"),
        ("mean_instability", "Mean Instability"),
        ("std_instability",  "Std Instability"),
        ("alert_rate",       "Alert Rate"),
    ]
    clip = {
        "peak_instability": 1.0,
        "mean_instability": 0.5,
        "std_instability":  0.5,
        "alert_rate":       0.5,
    }

    dead  = patient_summary[patient_summary["is_deceased"] == 1]
    alive = patient_summary[patient_summary["is_deceased"] == 0]

    fig, axes = plt.subplots(1, 4, figsize=(16, 5))
    for ax, (metric, label) in zip(axes, metrics):
        d_vals = np.clip(dead[metric].dropna().values,  0, clip[metric])
        a_vals = np.clip(alive[metric].dropna().values, 0, clip[metric])

        parts = ax.violinplot([a_vals, d_vals], positions=[1, 2],
                              showmedians=True, showextrema=False)

        colors = [COLOR_ALIVE, COLOR_DECEASED]
        hatches = ["//", "xx"]
        for pc, color, hatch in zip(parts["bodies"], colors, hatches):
            pc.set_facecolor(color)
            pc.set_alpha(0.7)
            pc.set_hatch(hatch)
            pc.set_edgecolor("black")
        parts["cmedians"].set_color("white")
        parts["cmedians"].set_linewidth(2)

        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Alive", "Deceased"])
        ax.set_title(label, fontsize=10)
        ax.grid(axis="y", alpha=0.3)

        if clip[metric] < 1.0:
            ax.set_ylim(0, clip[metric])
            ax.text(0.98, 0.97, f"clipped at {clip[metric]}",
                    transform=ax.transAxes, ha="right", va="top",
                    fontsize=7, color="gray")

        ymax = ax.get_ylim()[1]
        ax.text(1, ymax * 0.92, f"μ={a_vals.mean():.3f}",
                ha="center", fontsize=8, color=COLOR_ALIVE)
        ax.text(2, ymax * 0.92, f"μ={d_vals.mean():.3f}",
                ha="center", fontsize=8, color=COLOR_DECEASED)

    plt.suptitle("Instability Metrics: Alive vs Deceased (p < 0.001 for all)",
                 fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "violin_group_separation.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("  Saved: violin_group_separation.png")


# ── 5c. GROUP SEPARATION DENSITY PLOT ────────────────────────────────────

def plot_density_group(patient_summary):
    """
    Overlapping KDE density curves for each instability metric.
    Filled area shows distribution shape; gray overlap region shows
    how distinguishable the two groups are — less overlap = better.
    """
    metrics = [
        ("peak_instability", "Peak Instability"),
        ("mean_instability", "Mean Instability"),
        ("std_instability",  "Std Instability"),
        ("alert_rate",       "Alert Rate"),
    ]
    clip = {
        "peak_instability": 1.0,
        "mean_instability": 0.4,
        "std_instability":  0.4,
        "alert_rate":       0.4,
    }

    dead  = patient_summary[patient_summary["is_deceased"] == 1]
    alive = patient_summary[patient_summary["is_deceased"] == 0]

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    axes = axes.flatten()

    for ax, (metric, label) in zip(axes, metrics):
        d_vals = np.clip(dead[metric].dropna().values,  0, clip[metric])
        a_vals = np.clip(alive[metric].dropna().values, 0, clip[metric])

        x_grid = np.linspace(0, clip[metric], 500)

        try:
            kde_a = gaussian_kde(a_vals, bw_method=0.15)
            kde_d = gaussian_kde(d_vals, bw_method=0.15)
            dens_a = kde_a(x_grid)
            dens_d = kde_d(x_grid)

            # Normalize so area = 1
            dx = x_grid[1] - x_grid[0]
            dens_a /= dens_a.sum() * dx
            dens_d /= dens_d.sum() * dx

            # Filled distributions with hatching
            ax.fill_between(x_grid, dens_a, alpha=0.35,
                            color=COLOR_ALIVE,    label="Alive",
                            hatch="//", edgecolor=COLOR_ALIVE)
            ax.fill_between(x_grid, dens_d, alpha=0.35,
                            color=COLOR_DECEASED, label="Deceased",
                            hatch="xx", edgecolor=COLOR_DECEASED)

            # Outline curves with distinct line styles
            ax.plot(x_grid, dens_a, color=COLOR_ALIVE,    lw=1.8, linestyle="-")
            ax.plot(x_grid, dens_d, color=COLOR_DECEASED, lw=1.8, linestyle="--")

            # Gray overlap region
            overlap = np.minimum(dens_a, dens_d)
            ax.fill_between(x_grid, overlap, alpha=0.25,
                            color="gray", label="Overlap")

            # Median dashed lines
            ax.axvline(np.median(a_vals), color=COLOR_ALIVE,
                       linestyle="--", lw=1.2, alpha=0.8)
            ax.axvline(np.median(d_vals), color=COLOR_DECEASED,
                       linestyle="--", lw=1.2, alpha=0.8)

            # Annotate medians — position relative to y range
            ymax = max(dens_a.max(), dens_d.max())
            ax.text(np.median(a_vals), ymax * 0.97,
                    f"med={np.median(a_vals):.3f}",
                    color=COLOR_ALIVE, fontsize=7,
                    ha="center", va="top")
            ax.text(np.median(d_vals), ymax * 0.87,
                    f"med={np.median(d_vals):.3f}",
                    color=COLOR_DECEASED, fontsize=7,
                    ha="center", va="top")

        except Exception as e:
            ax.text(0.5, 0.5, f"KDE failed:\n{e}",
                    transform=ax.transAxes, ha="center", va="center",
                    fontsize=8, color="gray")

        ax.set_xlabel(label, fontsize=10)
        ax.set_xlim(0, clip[metric])
        ax.set_ylabel("Density", fontsize=9)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc="upper right")

        if clip[metric] < 1.0:
            ax.text(0.98, 0.02, f"x clipped at {clip[metric]}",
                    transform=ax.transAxes, ha="right", va="bottom",
                    fontsize=7, color="gray")

    fig.suptitle(
        "PEMDT Instability Metrics — Density: Alive vs Deceased\n"
        "(dashed lines = medians;  gray = overlap region)",
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "density_group_separation.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("  Saved: density_group_separation.png")


# ── 6. CALIBRATION PLOT ───────────────────────────────────────────────────

def calibration_analysis(patient_summary):
    y_true  = patient_summary["is_deceased"].values
    y_score = patient_summary["mean_instability"].values
    n_bins  = min(10, max(3, len(patient_summary) // 10))
    try:
        frac_pos, mean_pred = calibration_curve(
            y_true, y_score, n_bins=n_bins, strategy="uniform")
        brier = brier_score_loss(y_true, y_score)
        plt.figure(figsize=(6, 5))
        plt.plot(mean_pred, frac_pos, color="steelblue", lw=2,
                 marker="s", markersize=7, linestyle="-",
                 label=f"PEMDT  (Brier={brier:.3f})")
        plt.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
        plt.xlabel("Mean Predicted Probability")
        plt.ylabel("Fraction of Positives")
        plt.title("Calibration Plot")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "calibration.png", dpi=150)
        plt.close()
        print(f"  Calibration plot saved  (Brier={brier:.3f}, n_bins={n_bins})")
    except Exception as e:
        print(f"  Calibration plot skipped: {e}")


# ── 7. DISCHARGE DESTINATION ──────────────────────────────────────────────

def discharge_analysis(patient_summary):
    if "discharge_location" not in patient_summary.columns:
        return
    ps = patient_summary.dropna(subset=["discharge_location"]).copy()
    ps["home_discharge"] = (
        ps["discharge_location"].str.upper()
        .str.contains("HOME", na=False).astype(int))
    if ps["home_discharge"].nunique() < 2:
        return
    groups = ps.groupby("home_discharge")[
        ["peak_instability", "mean_instability", "std_instability"]
    ].agg(["mean", "std"]).round(3)
    print("\nDischarge destination analysis:")
    print(groups.to_string())
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, metric in zip(axes, ["peak_instability",
                                  "mean_instability",
                                  "std_instability"]):
        g0 = ps[ps["home_discharge"] == 0][metric].dropna().values
        g1 = ps[ps["home_discharge"] == 1][metric].dropna().values
        _, p = ttest_ind(g0, g1, equal_var=False)
        sig = "**" if p < 0.01 else ("*" if p < 0.05 else "ns")
        bp = ax.boxplot([g0, g1],
                   tick_labels=["Non-Home", "Home"],
                   patch_artist=True,
                   boxprops=dict(facecolor="lightyellow", alpha=0.8),
                   medianprops=dict(color="darkorange", lw=2))
        for patch, hatch in zip(bp["boxes"], ["//", ".."]):
            patch.set_hatch(hatch)
        ax.set_title(f"{metric}\np={p:.4f}  {sig}", fontsize=9)
        ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "discharge_analysis.png", dpi=150)
    plt.close()


# ── 8. LEAD-TIME ANALYSIS (with clipping) ────────────────────────────────

def alert_lead_time_analysis(dashboard):
    results = {}
    # Clip days per event — for visualization only, stats use full distribution
    clip_days = {
        "first_vasopressor_time": 7,
        "dod":                    30,
    }

    for event_col, event_label in [
        ("first_vasopressor_time", "Vasopressor Onset"),
        ("dod",                    "Death"),
    ]:
        if event_col not in dashboard.columns:
            continue
        print(f"  Computing lead-time for {event_label}...")
        df = dashboard.copy()
        if event_col == "dod":
            if "hospital_expire_flag" in df.columns:
                df = df[df["hospital_expire_flag"] == 1].copy()
            if "outtime" in df.columns:
                df["outtime"] = pd.to_datetime(df["outtime"])
                df = df[df[event_col] <= (df["outtime"] + pd.Timedelta(days=7))]
            df = df.dropna(subset=["dod"])
            print(f"  Filtering to stay-relevant deaths: "
                  f"{df['patient_id'].nunique()} patients")

        df["timestamp"] = pd.to_datetime(df["timestamp"],  errors="coerce")
        df[event_col]   = pd.to_datetime(df[event_col],    errors="coerce")

        df_alerts = df[
            (df["alert_triggered"] == True) &
            (df[event_col].notna()) &
            (df["timestamp"] <= df[event_col])
        ].copy()
        if df_alerts.empty:
            continue

        first_alerts = (df_alerts.groupby("patient_id")["timestamp"]
                        .min().reset_index()
                        .rename(columns={"timestamp": "first_alert_time"}))
        event_times  = (df[df[event_col].notna()]
                        .groupby("patient_id")[event_col]
                        .first().reset_index()
                        .rename(columns={event_col: "event_time"}))
        merged = first_alerts.merge(event_times, on="patient_id", how="inner")
        merged["lead_min"] = (
            merged["event_time"] - merged["first_alert_time"]
        ).dt.total_seconds() / 60
        lt = merged[merged["lead_min"] >= 0]["lead_min"].values
        if len(lt) == 0:
            continue

        results[event_label] = {
            "n":      len(lt),
            "median": round(np.median(lt), 1),
            "mean":   round(np.mean(lt),   1),
            "std":    round(np.std(lt),    1),
            "pct25":  round(np.percentile(lt, 25), 1),
            "pct75":  round(np.percentile(lt, 75), 1),
        }
        print(f"\nLead-time to {event_label}  (n={len(lt)}):")
        print(f"  Median : {np.median(lt):.1f} min")
        print(f"  Mean   : {np.mean(lt):.1f} min  ±  {np.std(lt):.1f}")
        print(f"  IQR    : [{np.percentile(lt,25):.1f}"
              f" - {np.percentile(lt,75):.1f}] min")

        # Clip for visualization only — stats computed on full lt above
        max_min    = clip_days[event_col] * 24 * 60
        lt_clipped = lt[lt <= max_min]
        n_clipped  = len(lt) - len(lt_clipped)

        plt.figure(figsize=(8, 4))
        plt.hist(lt_clipped, bins=30, edgecolor="black",
                 color="steelblue", alpha=0.8, hatch="//")
        plt.axvline(np.median(lt), color="red", linestyle="--", lw=2,
                    label=f"Median = {np.median(lt):.0f} min")
        plt.xlabel(f"Lead Time Before {event_label} (minutes)")
        plt.ylabel("Patient Count")
        plt.title(f"PEMDT Alert Lead-Time — {event_label}")
        if n_clipped > 0:
            plt.text(0.98, 0.95,
                     f"{n_clipped} outliers clipped "
                     f"(>{clip_days[event_col]} days)",
                     transform=plt.gca().transAxes,
                     ha="right", va="top", fontsize=8, color="gray")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"lead_time_{event_col}.png", dpi=150)
        plt.close()

    return results


# ── 9. INSTABILITY TRAJECTORIES ───────────────────────────────────────────

def plot_trajectories(dashboard, n=6):
    rng        = np.random.default_rng(RANDOM_SEED)
    dec_pids   = dashboard[dashboard["is_deceased"] == 1]["patient_id"].unique()
    alive_pids = dashboard[dashboard["is_deceased"] == 0]["patient_id"].unique()
    n_dec      = min(len(dec_pids),   n // 2)
    n_alive    = min(len(alive_pids), n - n_dec)
    sample     = np.concatenate([
        rng.choice(dec_pids,   n_dec,   replace=False),
        rng.choice(alive_pids, n_alive, replace=False),
    ])
    cols = 3
    rows = int(np.ceil(len(sample) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
    axes = axes.flatten()
    for ax, pid in zip(axes, sample):
        p_df  = dashboard[dashboard["patient_id"] == pid].sort_values("timestamp")
        dec   = int(p_df["is_deceased"].max())
        color = "crimson" if dec else "steelblue"
        lstyle = "--" if dec else "-"
        mkr = "x" if dec else "o"
        mkevery = max(1, len(p_df) // 8)
        ax.plot(p_df["timestamp"], p_df["unstable_probability"],
                color=color, lw=1.5, alpha=0.9,
                linestyle=lstyle, marker=mkr, markevery=mkevery, markersize=4,
                label="Deceased" if dec else "Alive")
        ax.axhline(ALERT_THRESHOLD, color="orange", linestyle="--", lw=1,
                   label=f"Threshold={ALERT_THRESHOLD}")
        if "first_vasopressor_time" in p_df.columns:
            vt = p_df["first_vasopressor_time"].iloc[0]
            if pd.notna(vt):
                ax.axvline(vt, color="purple", linestyle=":", lw=1.5,
                           label="Vasopressor")
        ax.set_title(f"{pid}  ({'Deceased' if dec else 'Alive'})", fontsize=9)
        ax.set_ylim(-0.05, 1.05)
        ax.set_ylabel("P(Unstable)", fontsize=8)
        ax.tick_params(axis="x", rotation=30, labelsize=7)
        ax.legend(fontsize=7, loc="upper left")
        ax.grid(alpha=0.3)
    for ax in axes[len(sample):]:
        ax.set_visible(False)
    plt.suptitle("PEMDT Patient Instability Trajectories")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "trajectories.png", dpi=150, bbox_inches="tight")
    plt.close()


# ── 10. DEMOGRAPHIC SUBGROUP ANALYSIS ────────────────────────────────────

def subgroup_analysis(patient_summary):
    print("\n--- Demographic Subgroup Analysis ---")
    rows = []
    if "gender" in patient_summary.columns:
        for g, grp in patient_summary.groupby("gender"):
            y_t = grp["is_deceased"].values
            y_s = grp["peak_instability"].values
            auc = roc_auc_score(y_t, y_s) if len(np.unique(y_t)) == 2 else np.nan
            rows.append({
                "subgroup":   f"Gender={g}",
                "n":          len(grp),
                "n_deceased": int(y_t.sum()),
                "mean_peak":  round(grp["peak_instability"].mean(), 3),
                "auc":        round(auc, 3) if not np.isnan(auc) else "n/a",
            })
    if "anchor_age" in patient_summary.columns:
        patient_summary["age_group"] = pd.cut(
            patient_summary["anchor_age"],
            bins=[0, 45, 65, 80, 120],
            labels=["<45", "45-65", "65-80", ">80"])
        for g, grp in patient_summary.groupby("age_group", observed=True):
            y_t = grp["is_deceased"].values
            y_s = grp["peak_instability"].values
            auc = roc_auc_score(y_t, y_s) if len(np.unique(y_t)) == 2 else np.nan
            rows.append({
                "subgroup":   f"Age={g}",
                "n":          len(grp),
                "n_deceased": int(y_t.sum()),
                "mean_peak":  round(grp["peak_instability"].mean(), 3),
                "auc":        round(auc, 3) if not np.isnan(auc) else "n/a",
            })
    if rows:
        sg_df = pd.DataFrame(rows)
        print(sg_df.to_string(index=False))
        sg_df.to_csv(OUTPUT_DIR / "subgroup_analysis.csv", index=False)
    return rows


# ── 10b. SUBGROUP AUC BAR CHART ───────────────────────────────────────────

def plot_subgroup_auc(patient_summary):
    """Bar chart of AUC by gender and age group."""
    rows = []
    if "gender" in patient_summary.columns:
        for g, grp in patient_summary.groupby("gender"):
            y_t = grp["is_deceased"].values
            y_s = grp["peak_instability"].values
            if len(np.unique(y_t)) == 2:
                rows.append({
                    "subgroup": f"{'Female' if g == 'F' else 'Male'}",
                    "auc":      round(roc_auc_score(y_t, y_s), 3),
                    "type":     "Gender",
                })
    if "anchor_age" in patient_summary.columns:
        if "age_group" not in patient_summary.columns:
            patient_summary["age_group"] = pd.cut(
                patient_summary["anchor_age"],
                bins=[0, 45, 65, 80, 120],
                labels=["<45", "45–65", "65–80", ">80"])
        for g, grp in patient_summary.groupby("age_group", observed=True):
            y_t = grp["is_deceased"].values
            y_s = grp["peak_instability"].values
            if len(np.unique(y_t)) == 2:
                rows.append({
                    "subgroup": f"Age {g}",
                    "auc":      round(roc_auc_score(y_t, y_s), 3),
                    "type":     "Age",
                })
    if not rows:
        print("  Subgroup AUC chart skipped — insufficient data.")
        return

    df     = pd.DataFrame(rows)
    colors  = ["#4C72B0" if t == "Gender" else "#DD8452" for t in df["type"]]
    hatches = ["//" if t == "Gender" else "xx" for t in df["type"]]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(df["subgroup"], df["auc"], color=colors,
                  edgecolor="black", linewidth=0.8, width=0.6)
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)

    for bar, val in zip(bars, df["auc"]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.003,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)

    overall_auc = roc_auc_score(
        patient_summary["is_deceased"].values,
        patient_summary["peak_instability"].values)
    ax.axhline(overall_auc, color="red", linestyle="--", lw=1.5,
               label=f"Overall AUC = {overall_auc:.3f}")

    ax.set_ylim(0.55, 0.75)
    ax.set_ylabel("ROC-AUC", fontsize=11)
    ax.set_title("PEMDT Subgroup AUC by Gender and Age", fontsize=12)
    ax.grid(axis="y", alpha=0.3)

    legend_elements = [
        mpatches.Patch(facecolor="#4C72B0", hatch="//", edgecolor="black", label="Gender"),
        mpatches.Patch(facecolor="#DD8452", hatch="xx", edgecolor="black", label="Age Group"),
        plt.Line2D([0], [0], color="red", linestyle="--",
                   lw=1.5, label=f"Overall AUC = {overall_auc:.3f}"),
    ]
    ax.legend(handles=legend_elements, fontsize=9)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "subgroup_auc.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  Saved: subgroup_auc.png")


# ── 11. SAVE RESULTS ──────────────────────────────────────────────────────

def save_results(patient_summary, metrics, group_results, lead_time_results):
    patient_summary.to_csv(OUTPUT_DIR / "patient_level_validation.csv", index=False)
    flat = {**metrics}
    for metric, vals in group_results.items():
        for k, v in vals.items():
            flat[f"{metric}_{k}"] = v
    for event, vals in lead_time_results.items():
        for k, v in vals.items():
            flat[f"leadtime_{event.replace(' ', '_')}_{k}"] = v
    pd.DataFrame([flat]).to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
    print("\n=== Final Metrics Summary ===")
    for k, v in flat.items():
        print(f"  {k:45s}: {v}")


# ── MAIN ──────────────────────────────────────────────────────────────────

def run_validation():
    print("\n========== PEMDT v2 Validation ==========\n")
    dashboard       = load_data()
    patient_summary = aggregate_to_patient(dashboard)

    print("\n--- ROC / PR Curves ---")
    metrics = plot_roc_pr(patient_summary)

    print("\n--- Group Comparison ---")
    group_results = group_comparison(patient_summary)

    print("\n--- Group Separation Violin ---")
    plot_violin_group(patient_summary)

    print("\n--- Group Separation Density ---")
    plot_density_group(patient_summary)

    print("\n--- Calibration ---")
    calibration_analysis(patient_summary)

    print("\n--- Discharge Analysis ---")
    discharge_analysis(patient_summary)

    print("\n--- Clinical Event Lead-Time ---")
    lead_time_results = alert_lead_time_analysis(dashboard)

    print("\n--- Instability Trajectories ---")
    plot_trajectories(dashboard)

    print("\n--- Subgroup Analysis ---")
    subgroup_analysis(patient_summary)

    print("\n--- Subgroup AUC Chart ---")
    plot_subgroup_auc(patient_summary)

    save_results(patient_summary, metrics, group_results, lead_time_results)
    print(f"\n✅ Validation complete — outputs in: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    run_validation()


========== PEMDT v2 Validation ==========

Dashboard shape: (9890935, 36)
Columns: ['subject_id', 'stay_id', 'timestamp', 'DBP', 'GCS_total', 'HR_bpm', 'MAP', 'Resp_rate', 'SBP', 'SpO2', 'Temp_C', 'HRV_proxy', 'patient_id', 'Creatinine', 'Hemoglobin', 'Lactate', 'Sodium', 'WBC', 'activity_state', 'activity_intensity', 'activity', 'has_behavioral_data', 'race', 'intime', 'outtime', 'los', 'first_vasopressor_time', 'unstable_probability', 'alert_triggered', 'clinical_explanation', 'is_deceased', 'dod', 'anchor_age', 'gender', 'discharge_location', 'hospital_expire_flag']

Mortality distribution (patient level):
is_deceased
0    43424
1    21267
Name: count, dtype: int64

Patient-level summary — 64691 patients, 21267 deaths (32.9%)

--- ROC / PR Curves ---

ROC-AUC : 0.687  95% CI [0.683-0.692]
PR-AUC  : 0.525  95% CI [0.518-0.533]
Brier   : 0.261

--- Group Comparison ---
peak_instability           dead=0.787  alive=0.557  p(t)=0.0000  p(U)=0.0000  **
mean_instability           dead=0.1

## Cell 4 — Calibrated Validation (Platt Scaling)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             brier_score_loss, roc_curve, precision_recall_curve)
from sklearn.calibration import calibration_curve
from scipy.stats import bootstrap

OUTPUT_DIR = Path("validation_output calibrated")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Load ──────────────────────────────────────────────────────────────────
print("Loading dashboard...")
dash = pd.read_csv("PEMDT_dashboard_cohort.csv", low_memory=False)
print(f"  Shape: {dash.shape}")

# ── Patient summary ───────────────────────────────────────────────────────
ps = dash.groupby("patient_id").agg(
    mean_instability  = ("unstable_probability", "mean"),
    peak_instability  = ("unstable_probability", "max"),
    std_instability   = ("unstable_probability", "std"),
    alert_rate        = ("alert_triggered",       "mean"),
    is_deceased       = ("is_deceased",           "first"),
    anchor_age        = ("anchor_age",            "first"),
).dropna(subset=["is_deceased"]).reset_index()

ps["is_deceased"] = ps["is_deceased"].astype(int)
ps["std_instability"] = ps["std_instability"].fillna(0)

print(f"  Patients: {len(ps):,}  |  Deaths: {ps['is_deceased'].sum():,}")

# ── Logistic calibration (Platt scaling) ──────────────────────────────────
from sklearn.preprocessing import StandardScaler

FEATURES = ["mean_instability", "peak_instability", "alert_rate", "anchor_age"]
X = ps[FEATURES].values
y = ps["is_deceased"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_scaled, y)
ps["calibrated_risk"] = lr.predict_proba(X_scaled)[:, 1]

print("\n  Calibration coefficients:")
for feat, coef in zip(FEATURES, lr.coef_[0]):
    print(f"    {feat:<25}: {coef:+.4f}")

raw_auc  = roc_auc_score(y, ps["mean_instability"])
cal_auc  = roc_auc_score(y, ps["calibrated_risk"])
raw_brier = brier_score_loss(y, ps["mean_instability"])
cal_brier = brier_score_loss(y, ps["calibrated_risk"])
print(f"\n  Raw HMM    — AUC: {raw_auc:.3f}  Brier: {raw_brier:.3f}")
print(f"  Calibrated — AUC: {cal_auc:.3f}  Brier: {cal_brier:.3f}")
print(f"  AUC gain: +{cal_auc - raw_auc:.3f}")

# ── Bootstrap CI ─────────────────────────────────────────────────────────
def bootstrap_auc(y_true, y_score, n_boot=1000):
    rng = np.random.default_rng(42)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if y_true[idx].sum() < 2:
            continue
        scores.append(roc_auc_score(y_true[idx], y_score[idx]))
    return np.percentile(scores, [2.5, 97.5])

ci = bootstrap_auc(y, ps["calibrated_risk"].values)
pr_auc = average_precision_score(y, ps["calibrated_risk"])

print(f"\n  Calibrated AUC: {cal_auc:.3f}  95% CI [{ci[0]:.3f}–{ci[1]:.3f}]")
print(f"  Calibrated PR-AUC: {pr_auc:.3f}")
print(f"  Calibrated Brier:  {cal_brier:.3f}")

# ============================================================
# PLOT 1 — ROC + PR curves (calibrated vs raw overlay)
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(7, 10))
fig.suptitle("Calibrated PEMDT — ROC and PR Curves", fontsize=13, fontweight="bold")

# ROC
fpr_raw,  tpr_raw,  _ = roc_curve(y, ps["mean_instability"])
fpr_cal,  tpr_cal,  _ = roc_curve(y, ps["calibrated_risk"])

ax = axes[0]
ax.plot(fpr_raw, tpr_raw, color="#95A5A6", lw=1.5, linestyle="--",
        marker="^", markevery=max(1, len(fpr_raw)//8), markersize=5,
        label=f"Raw HMM  AUC={raw_auc:.3f}")
ax.plot(fpr_cal, tpr_cal, color="#2C6FAC", lw=2.2, linestyle="-",
        marker="o", markevery=max(1, len(fpr_cal)//8), markersize=5,
        label=f"Calibrated  AUC={cal_auc:.3f}  [{ci[0]:.3f}–{ci[1]:.3f}]")
ax.plot([0,1],[0,1], "k--", lw=0.8, alpha=0.4)
ax.fill_between(fpr_cal, tpr_cal, alpha=0.08, color="#2C6FAC")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("ROC Curve", fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_facecolor("#FAFAFA")

# PR
prec_raw, rec_raw, _ = precision_recall_curve(y, ps["mean_instability"])
prec_cal, rec_cal, _ = precision_recall_curve(y, ps["calibrated_risk"])
pr_auc_raw = average_precision_score(y, ps["mean_instability"])

ax = axes[1]
ax.plot(rec_raw, prec_raw, color="#95A5A6", lw=1.5, linestyle="--",
        marker="^", markevery=max(1, len(rec_raw)//8), markersize=5,
        label=f"Raw HMM  PR-AUC={pr_auc_raw:.3f}")
ax.plot(rec_cal, prec_cal, color="#1ABC9C", lw=2.2, linestyle="-",
        marker="s", markevery=max(1, len(rec_cal)//8), markersize=5,
        label=f"Calibrated  PR-AUC={pr_auc:.3f}")
ax.fill_between(rec_cal, prec_cal, alpha=0.08, color="#1ABC9C")
ax.axhline(y.mean(), color="gray", lw=0.8, linestyle=":", label="Baseline (prevalence)")
ax.set_xlabel("Recall", fontsize=11)
ax.set_ylabel("Precision", fontsize=11)
ax.set_title("Precision-Recall Curve", fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_facecolor("#FAFAFA")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_pr_calibrated.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: roc_pr_calibrated.png")


# ============================================================
# PLOT 2 — Reliability / calibration plot (calibrated vs raw)
# ============================================================
fig, ax = plt.subplots(figsize=(7, 6))

frac_raw, mean_raw = calibration_curve(y, ps["mean_instability"],  n_bins=10, strategy="uniform")
frac_cal, mean_cal = calibration_curve(y, ps["calibrated_risk"],   n_bins=10, strategy="uniform")

ax.plot(mean_raw, frac_raw, color="#95A5A6", lw=1.8, linestyle="--",
        marker="^", markersize=7,
        label=f"Raw HMM  (Brier={raw_brier:.3f})")
ax.plot(mean_cal, frac_cal, color="#2C6FAC", lw=2.2, linestyle="-",
        marker="s", markersize=7,
        label=f"Calibrated  (Brier={cal_brier:.3f})")
ax.plot([0,1],[0,1], "k--", lw=1, label="Perfect calibration")
ax.fill_between(mean_cal, frac_cal, mean_cal, alpha=0.1, color="#2C6FAC")

ax.set_xlabel("Mean Predicted Probability", fontsize=11)
ax.set_ylabel("Fraction of Positives",      fontsize=11)
ax.set_title("Reliability Plot: Raw vs Calibrated HMM", fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_facecolor("#FAFAFA")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "calibration_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: calibration_plot.png")


# ============================================================
# PLOT 3 — Calibration coefficient bar chart
# ============================================================
fig, ax = plt.subplots(figsize=(7, 5))

coef_labels = ["Mean\nInstability", "Peak\nInstability", "Alert\nRate", "Anchor\nAge"]
coefs       = lr.coef_[0]
colors      = ["#2C6FAC" if c > 0 else "#C0392B" for c in coefs]

bars = ax.bar(coef_labels, coefs, color=colors,
              edgecolor="black", linewidth=0.8, width=0.5)
for bar, c in zip(bars, coefs):
    bar.set_hatch("//" if c > 0 else "xx")
ax.axhline(0, color="black", linewidth=1)

for bar, val in zip(bars, coefs):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + (0.01 if val >= 0 else -0.03),
            f"{val:+.3f}", ha="center",
            va="bottom" if val >= 0 else "top", fontsize=11, fontweight="bold")

ax.set_ylabel("Standardised Coefficient", fontsize=11)
ax.set_title("Logistic Calibration Layer — Feature Coefficients\n"
             "(Platt scaling on patient-level HMM summary features)", fontsize=11)
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.set_facecolor("#FAFAFA")

pos_patch = mpatches.Patch(facecolor="#2C6FAC", hatch="//", edgecolor="black", label="Positive — increases predicted risk")
neg_patch = mpatches.Patch(facecolor="#C0392B", hatch="xx", edgecolor="black", label="Negative — decreases predicted risk")
ax.legend(handles=[pos_patch, neg_patch], fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "calibration_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: calibration_coefficients.png")


# ============================================================
# PLOT 4 — Raw vs calibrated risk scatter (coloured by outcome)
# ============================================================
fig, ax = plt.subplots(figsize=(7, 6))

alive = ps[ps["is_deceased"] == 0]
dead  = ps[ps["is_deceased"] == 1]

ax.scatter(alive["mean_instability"], alive["calibrated_risk"],
           alpha=0.15, s=12, color="#3498DB", label="Survived", rasterized=True, marker="o")
ax.scatter(dead["mean_instability"],  dead["calibrated_risk"],
           alpha=0.25, s=12, color="#E74C3C", label="Died",      rasterized=True, marker="x")
ax.plot([0,1],[0,1], "k--", lw=0.8, alpha=0.5, label="No change line")

ax.set_xlabel("Raw HMM Score (mean_instability)", fontsize=11)
ax.set_ylabel("Calibrated Risk Score",            fontsize=11)
ax.set_title("Raw vs Calibrated Risk per Patient\n"
             "Points above diagonal = calibration increased risk estimate", fontsize=11)
ax.legend(fontsize=9, markerscale=4)
ax.grid(alpha=0.3)
ax.set_facecolor("#FAFAFA")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "raw_vs_calibrated_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: raw_vs_calibrated_scatter.png")

print(f"\n✅ Calibrated validation complete — 4 unique plots in: {OUTPUT_DIR}")
print(f"   roc_pr_calibrated.png        — ROC/PR with raw overlay")
print(f"   calibration_plot.png         — Reliability curve raw vs calibrated")
print(f"   calibration_coefficients.png — Coefficient bar chart")
print(f"   raw_vs_calibrated_scatter.png — Score transformation scatter")

Loading dashboard...
  Shape: (9890935, 36)
  Patients: 64,691  |  Deaths: 21,267

  Calibration coefficients:
    mean_instability         : +0.6040
    peak_instability         : +0.3777
    alert_rate               : -0.1661
    anchor_age               : +0.6304

  Raw HMM    — AUC: 0.687  Brier: 0.261
  Calibrated — AUC: 0.740  Brier: 0.186
  AUC gain: +0.053

  Calibrated AUC: 0.740  95% CI [0.736–0.744]
  Calibrated PR-AUC: 0.579
  Calibrated Brier:  0.186
✅ Saved: roc_pr_calibrated.png
✅ Saved: calibration_plot.png
✅ Saved: calibration_coefficients.png
✅ Saved: raw_vs_calibrated_scatter.png

✅ Calibrated validation complete — 4 unique plots in: validation_output calibrated
   roc_pr_calibrated.png        — ROC/PR with raw overlay
   calibration_plot.png         — Reliability curve raw vs calibrated
   calibration_coefficients.png — Coefficient bar chart
   raw_vs_calibrated_scatter.png — Score transformation scatter


## Cell 5 — Standalone LR Benchmark

In [4]:
# ============================================================
# PEMDT v2 — Standalone Logistic Regression Benchmark
#
# NOTE: This is a SEPARATE model from the logistic calibration
# layer in Cell 4. The calibration layer (4 features, HMM-derived)
# post-processes HMM output. This benchmark (6 features, including
# std_instability and length of stay) is trained independently on
# a 70/30 stratified split without access to calibration outputs.
# It serves as an upper-bound reference for discriminative performance.
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.metrics import roc_curve
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_DIR = Path("validation_output calibrated")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Loading patient_summary.csv ...")
df = pd.read_csv("patient_summary.csv")

features = [
    "mean_instability",
    "peak_instability",
    "std_instability",
    "alert_rate",
    "anchor_age",
    "los"
]

data = pd.concat([df[features], df["is_deceased"]], axis=1).dropna()
X = data[features]
y = data["is_deceased"]
print(f"Total patients used: {len(X)}  |  Deaths: {y.sum()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc  = average_precision_score(y_test, y_prob)
brier   = brier_score_loss(y_test, y_prob)

print(f"\n--- Standalone LR Performance ---")
print(f"ROC-AUC : {roc_auc:.3f}")
print(f"PR-AUC  : {pr_auc:.3f}")
print(f"Brier   : {brier:.3f}")

coef_df = pd.DataFrame({
    "Feature":     features,
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient", ascending=False)

print("\n--- Feature Coefficients (standardised) ---")
print(coef_df.to_string(index=False))

fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Standalone LR (AUC = {roc_auc:.3f})")
plt.plot([0,1],[0,1],"k--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Standalone LR ROC Curve"); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lr_benchmark_roc.png", dpi=150)
plt.close()
print(f"\n✅ LR benchmark complete.")


Loading patient_summary.csv ...
Total patients used: 64646  |  Deaths: 21231

--- Standalone LR Performance ---
ROC-AUC : 0.740
PR-AUC  : 0.576
Brier   : 0.185

--- Feature Coefficients (standardised) ---
         Feature  Coefficient
      anchor_age     0.640797
mean_instability     0.417365
 std_instability     0.323217
peak_instability     0.166720
             los     0.138498
      alert_rate    -0.122053

✅ LR benchmark complete.


## Cell 6 — Experiment Sweep (180 configurations)

In [1]:
# ============================================================
# PEMDT v2 — Experimental Design  (Conference-Ready)
# ============================================================
#
#  180 experiment combinations:
#    transition_aggressiveness  ×  3
#    activity_weight            ×  3
#    alert_threshold            ×  4
#    anomaly_injection_rate     ×  5
#
#  Bayesian estimator matches pipeline exactly:
#    - Age-adjusted prior + transition (4 brackets)
#    - Global z-score normalization
#    - Clinical feature weighting
#    - Log-sum-exp numerical stability
#    - Full feature set: vitals + labs + activity
#
#  No calibration here — calibration is learned in validation.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import itertools
import warnings
from pathlib import Path
from scipy.stats import norm
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

DASHBOARD_FILE = "PEMDT_dashboard_cohort.csv"
OUTPUT_DIR     = Path("experiment_output")
OUTPUT_DIR.mkdir(exist_ok=True)

BOOTSTRAP_N = 500
RANDOM_SEED = 42

# ============================================================
# GLOBAL STATS — must match pipeline exactly
# ============================================================

GLOBAL_STATS = {
    "SBP":        (120, 20),
    "HRV_proxy":  (0.15, 0.10),
    "SpO2":       (97,   3),
    "Resp_rate":  (18,   5),
    "MAP":        (80,   15),
    "Lactate":    (2.0,  1.5),
    "WBC":        (10,   5),
    "Creatinine": (1.2,  0.8),
}

FEATURE_WEIGHTS = {
    "SBP":        1.3,
    "HRV_proxy":  1.5,
    "SpO2":       1.4,
    "Resp_rate":  1.2,
    "MAP":        1.4,
    "Lactate":    1.6,
    "WBC":        1.0,
    "Creatinine": 1.1,
}

UNSTABLE_Z = {
    "SBP":        -2.0,
    "HRV_proxy":  -2.0,
    "SpO2":       -2.0,
    "Resp_rate":  +2.0,
    "MAP":        -2.0,
    "Lactate":    +2.0,
    "WBC":        +2.0,
    "Creatinine": +2.0,
}

VALID_RANGES = {
    "SBP":        (40,  260),
    "HRV_proxy":  (0,   1),
    "SpO2":       (50,  100),
    "Resp_rate":  (4,   60),
    "MAP":        (20,  200),
    "Lactate":    (0.1, 30),
    "WBC":        (0.1, 200),
    "Creatinine": (0.1, 30),
}

# ============================================================
# PARAMETER GRID  (3 × 3 × 4 × 5 = 180 runs)
# ============================================================

TRANSITION_LEVELS = {
    "conservative": 0.03,
    "moderate":     0.07,
    "aggressive":   0.15,
}

ACTIVITY_WEIGHT_LEVELS = {
    "low":    0.5,
    "medium": 1.0,
    "high":   2.0,
}

ALERT_THRESHOLDS = [0.3, 0.4, 0.5, 0.6]

ANOMALY_RATES = [0.0, 0.05, 0.10, 0.15, 0.20]


# ============================================================
# PARAMETRISED BAYESIAN ESTIMATOR
# Matches pipeline bayesian_health_state_patient exactly,
# with transition_aggressiveness and activity_weight as params.
# ============================================================

def run_bayesian_parametrised(
    df: pd.DataFrame,
    transition_aggressiveness: float,
    activity_weight: float,
    anomaly_rate: float = 0.0,
    rng: np.random.Generator = None,
) -> pd.DataFrame:
    df = df.copy().sort_values("timestamp").reset_index(drop=True)
    if rng is None:
        rng = np.random.default_rng(RANDOM_SEED)

    # ── Anomaly injection ─────────────────────────────────────────────────
    df["anomaly_injected"] = False
    if anomaly_rate > 0:
        n_inject   = max(1, int(len(df) * anomaly_rate))
        inject_idx = rng.choice(len(df), n_inject, replace=False)
        if "SBP"       in df.columns:
            df.loc[inject_idx, "SBP"]       = df.loc[inject_idx, "SBP"].fillna(120) + 65
        if "HRV_proxy" in df.columns:
            df.loc[inject_idx, "HRV_proxy"] = 0.02
        if "Lactate"   in df.columns:
            df.loc[inject_idx, "Lactate"]   = df.loc[inject_idx, "Lactate"].fillna(1.0) + 5.0
        df.loc[inject_idx, "anomaly_injected"] = True

    # ── Age extraction ────────────────────────────────────────────────────
    age = df["anchor_age"].iloc[0] if "anchor_age" in df.columns else 65
    age = age if pd.notna(age) and 18 < age < 110 else 65

    # ── Age-adjusted prior + base transition ──────────────────────────────
    # transition_aggressiveness overrides the [0,1] cell (stable→unstable rate)
    # The age bracket shapes the recovery rate [1,0] and initial belief
    if age < 45:
        belief_t   = np.array([0.97, 0.03])
        recovery   = 0.20
    elif age < 65:
        belief_t   = np.array([0.95, 0.05])
        recovery   = 0.15
    elif age < 80:
        belief_t   = np.array([0.92, 0.08])
        recovery   = 0.10
    else:
        belief_t   = np.array([0.88, 0.12])
        recovery   = 0.08

    base_trans = np.array([
        [1 - transition_aggressiveness, transition_aggressiveness],
        [recovery,                       1 - recovery],
    ])

    # ── Vectorised likelihoods ────────────────────────────────────────────
    n = len(df)
    log_l_s = np.zeros(n)
    log_l_u = np.zeros(n)

    for feat in FEATURE_WEIGHTS:
        if feat == "activity" or feat not in df.columns:
            continue
        lo, hi = VALID_RANGES.get(feat, (-np.inf, np.inf))
        vals   = df[feat].values.astype(float)
        mask   = np.isfinite(vals) & (vals > lo) & (vals < hi)
        if not mask.any():
            continue
        mu, sd     = GLOBAL_STATS[feat]
        z          = np.where(mask, (vals - mu) / sd, 0.0)
        w          = FEATURE_WEIGHTS[feat]
        z_unstable = UNSTABLE_Z[feat]
        ls = w * (-0.5 * z**2 - np.log(sd * np.sqrt(2 * np.pi)))
        lu = w * (-0.5 * ((z - z_unstable) / 1.5)**2 - np.log(1.5 * sd * np.sqrt(2 * np.pi)))
        log_l_s = np.where(mask, log_l_s + ls, log_l_s)
        log_l_u = np.where(mask, log_l_u + lu, log_l_u)

    # ── Activity context (scaled by activity_weight param) ────────────────
    if "has_behavioral_data" in df.columns:
        has_beh = df["has_behavioral_data"].fillna(False).astype(bool).values
        act     = df["activity_state"].fillna("Unknown").values if "activity_state" in df.columns else np.full(n, "Unknown")

        active_mask   = has_beh & np.isin(act, ["Active", "Cooking", "Walking"])
        inactive_mask = has_beh & ~active_mask

        raw_s = np.where(active_mask, 0.7, np.where(inactive_mask, 0.9, 1.0))
        raw_u = np.where(active_mask, 0.3, np.where(inactive_mask, 0.1, 1.0))

        # Scale contribution by activity_weight parameter
        p_act_s = np.clip(1 - activity_weight * (1 - raw_s), 0.01, 0.99)
        p_act_u = np.clip(1 - activity_weight * (1 - raw_u), 0.01, 0.99)

        log_l_s += np.where(has_beh, np.log(p_act_s + 1e-300), 0.0)
        log_l_u += np.where(has_beh, np.log(p_act_u + 1e-300), 0.0)

    # ── Forward filtering ─────────────────────────────────────────────────
    health_probs = np.empty(n)

    for i in range(n):
        # Adaptive momentum: once instability > 0.4, increase persistence
        transition_matrix = base_trans.copy()
        if belief_t[1] > 0.4:
            transition_matrix[1, 1] = 0.95
            transition_matrix[1, 0] = 0.05

        belief_pred = belief_t @ transition_matrix

        log_post_s = np.log(belief_pred[0] + 1e-300) + log_l_s[i]
        log_post_u = np.log(belief_pred[1] + 1e-300) + log_l_u[i]

        log_max  = max(log_post_s, log_post_u)
        post_s   = np.exp(log_post_s - log_max)
        post_u   = np.exp(log_post_u - log_max)

        belief_t = np.array([post_s, post_u]) / (post_s + post_u + 1e-9)
        health_probs[i] = belief_t[1]

    df["unstable_probability"] = health_probs
    return df


# ============================================================
# LEAD TIME AGAINST REAL VASOPRESSOR EVENT
# ============================================================

def compute_real_lead_time(df: pd.DataFrame, threshold: float):
    if "first_vasopressor_time" not in df.columns:
        return None
    vaso_time = pd.to_datetime(df["first_vasopressor_time"].iloc[0], errors="coerce")
    if pd.isna(vaso_time):
        return None
    pre    = df[pd.to_datetime(df["timestamp"], errors="coerce") <= vaso_time]
    alerts = pre[pre["unstable_probability"] > threshold]
    if alerts.empty:
        return 0.0
    first_alert = pd.to_datetime(alerts["timestamp"].min(), errors="coerce")
    return (vaso_time - first_alert).total_seconds() / 60


# ============================================================
# BOOTSTRAP AUC
# ============================================================

def bootstrap_auc_fast(y_true, y_score, n=BOOTSTRAP_N, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    if len(np.unique(y_true)) < 2:
        return np.nan, np.nan, np.nan
    pt   = roc_auc_score(y_true, y_score)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            vals.append(roc_auc_score(y_true[idx], y_score[idx]))
        except Exception:
            continue
    lo, hi = np.percentile(vals, [2.5, 97.5]) if vals else (np.nan, np.nan)
    return pt, lo, hi


# ============================================================
# SINGLE EXPERIMENT RUN
# ============================================================

def run_single_experiment(
    fused_df: pd.DataFrame,
    trans_name: str, trans_val: float,
    act_name: str,   act_val: float,
    threshold: float,
    anomaly_rate: float,
) -> dict:

    rng = np.random.default_rng(RANDOM_SEED)
    all_results      = []
    real_lead_times  = []
    synth_lead_times = []

    for pid, group in fused_df.groupby("patient_id"):
        result = run_bayesian_parametrised(
            group, trans_val, act_val, anomaly_rate, rng
        )
        result["alert_triggered"] = result["unstable_probability"] > threshold
        all_results.append(result)

        # Real lead time
        rlt = compute_real_lead_time(result, threshold)
        if rlt is not None and rlt > 0:
            real_lead_times.append(rlt)

        # Synthetic anomaly lead time
        if anomaly_rate > 0:
            df_r     = result.reset_index(drop=True)
            anom_idx = df_r[df_r["anomaly_injected"] == True].index
            if len(anom_idx) > 0:
                first_anom = anom_idx.min()
                early      = df_r[
                    (df_r.index <= first_anom) &
                    (df_r["unstable_probability"] > threshold)
                ]
                if not early.empty:
                    synth_lead_times.append(float(first_anom - early.index.min()))

    combined = pd.concat(all_results, ignore_index=True)

    # ── Patient-level aggregation ─────────────────────────────────────────
    pat_agg = (
        combined.groupby("patient_id")
        .agg(
            peak_instability = ("unstable_probability", "max"),
            mean_instability = ("unstable_probability", "mean"),  # key metric
            alert_rate_val   = ("alert_triggered",       "mean"),
        )
        .reset_index()
    )

    if "is_deceased" in combined.columns:
        dec     = combined.groupby("patient_id")["is_deceased"].max().reset_index()
        pat_agg = pat_agg.merge(dec, on="patient_id", how="left")
    else:
        pat_agg["is_deceased"] = 0

    pat_agg["is_deceased"] = pat_agg["is_deceased"].fillna(0).astype(int)

    # ── AUC — use mean_instability (stronger predictor per calibration) ───
    pa = pat_agg.dropna(subset=["is_deceased"])

    auc_pt, auc_lo, auc_hi = bootstrap_auc_fast(
        pa["is_deceased"].values,
        pa["mean_instability"].values,
    )

    # PR-AUC
    pr_auc = np.nan
    if len(np.unique(pa["is_deceased"])) == 2:
        try:
            pr_auc = average_precision_score(
                pa["is_deceased"].values,
                pa["mean_instability"].values,
            )
        except Exception:
            pass

    # False alarm rate (non-anomaly windows)
    non_anom         = combined[~combined.get("anomaly_injected", pd.Series(False, index=combined.index))]
    false_alarm_rate = (non_anom["unstable_probability"] > threshold).mean()

    # Sensitivity (anomaly windows)
    sensitivity = np.nan
    if anomaly_rate > 0:
        anom_wins = combined[combined.get("anomaly_injected", pd.Series(False, index=combined.index)) == True]
        if len(anom_wins) > 0:
            sensitivity = (anom_wins["unstable_probability"] > threshold).mean()

    # ── Behavioral context split ──────────────────────────────────────────
    auc_beh, auc_icu = np.nan, np.nan
    if "has_behavioral_data" in combined.columns:
        for flag, label in [(True, "beh"), (False, "icu")]:
            sub_pids = (combined[combined["has_behavioral_data"] == flag]
                        ["patient_id"].unique())
            sub_pat  = pat_agg[pat_agg["patient_id"].isin(sub_pids)]
            if len(sub_pat) > 5 and sub_pat["is_deceased"].nunique() == 2:
                try:
                    val = roc_auc_score(
                        sub_pat["is_deceased"].values,
                        sub_pat["mean_instability"].values,
                    )
                    if label == "beh": auc_beh = val
                    else:              auc_icu = val
                except Exception:
                    pass

    return {
        "transition":              trans_name,
        "activity_wt":             act_name,
        "threshold":               threshold,
        "anomaly_rate":            anomaly_rate,
        "auc":                     round(auc_pt, 4) if not np.isnan(auc_pt) else np.nan,
        "auc_ci_lo":               round(auc_lo, 4) if not np.isnan(auc_lo) else np.nan,
        "auc_ci_hi":               round(auc_hi, 4) if not np.isnan(auc_hi) else np.nan,
        "pr_auc":                  round(pr_auc, 4) if not np.isnan(pr_auc) else np.nan,
        "alert_rate":              round(pat_agg["alert_rate_val"].mean(), 4),
        "false_alarm_rate":        round(false_alarm_rate, 4),
        "sensitivity":             round(sensitivity, 4) if not np.isnan(sensitivity) else np.nan,
        "synth_lead_time_windows": round(np.mean(synth_lead_times), 2) if synth_lead_times else np.nan,
        "real_lead_time_min":      round(np.mean(real_lead_times), 1) if real_lead_times else np.nan,
        "auc_with_behavioral":     round(auc_beh, 4) if not np.isnan(auc_beh) else np.nan,
        "auc_icu_only":            round(auc_icu, 4) if not np.isnan(auc_icu) else np.nan,
    }


# ============================================================
# FULL 180-RUN SWEEP
# ============================================================

def run_all_experiments(fused_df: pd.DataFrame) -> pd.DataFrame:
    grid = list(itertools.product(
        TRANSITION_LEVELS.items(),
        ACTIVITY_WEIGHT_LEVELS.items(),
        ALERT_THRESHOLDS,
        ANOMALY_RATES,
    ))

    print(f"\nTotal experiment runs: {len(grid)}  "
          f"({len(TRANSITION_LEVELS)} trans × "
          f"{len(ACTIVITY_WEIGHT_LEVELS)} act × "
          f"{len(ALERT_THRESHOLDS)} thresh × "
          f"{len(ANOMALY_RATES)} anomaly)")

    results = []
    for i, ((tn, tv), (an, av), thresh, anomaly) in enumerate(grid):
        if (i + 1) % 30 == 0 or i == 0:
            print(f"  Run {i+1:3d}/{len(grid)} — "
                  f"trans={tn:12s}  act={an:6s}  "
                  f"thresh={thresh}  anomaly={anomaly}")
        r = run_single_experiment(
            fused_df, tn, tv, an, av, thresh, anomaly
        )
        results.append(r)

    df_out = pd.DataFrame(results)
    df_out.to_csv(OUTPUT_DIR / "experiment_results.csv", index=False)
    print(f"\n✅ All {len(results)} runs complete.")
    return df_out


# ============================================================
# ANALYSIS AND PLOTS
# ============================================================

def analyse_experiments(results: pd.DataFrame):
    print("\n=== Experiment Analysis ===")

    r_valid = results[results["auc"].notna()]

    # ── Best config by AUC ────────────────────────────────────────────────
    if not r_valid.empty:
        best = r_valid.loc[r_valid["auc"].idxmax()]
        print("\n Best config by AUC:")
        for k, v in best.items():
            print(f"   {k}: {v}")

    # ── Best config by sensitivity/false-alarm balance ────────────────────
    r2 = results.dropna(subset=["sensitivity"]).copy()
    if not r2.empty:
        r2["balance"] = (
            2 * r2["sensitivity"] * (1 - r2["false_alarm_rate"])
        ) / (r2["sensitivity"] + (1 - r2["false_alarm_rate"]) + 1e-6)
        best_bal = r2.loc[r2["balance"].idxmax()]
        print("\n Best config by Sensitivity/FalseAlarm balance:")
        for k, v in best_bal.items():
            print(f"   {k}: {v}")

    # ── Plot 1: AUC heatmap (anomaly_rate=0) ─────────────────────────────
    r0 = results[results["anomaly_rate"] == 0.0]
    if not r0.empty and r0["auc"].notna().any():
        pivot = r0.pivot_table(
            values="auc", index="transition",
            columns="activity_wt", aggfunc="mean"
        )
        fig, ax = plt.subplots(figsize=(7, 5))
        vals_clean = pivot.values[~np.isnan(pivot.values)]
        im = ax.imshow(
            pivot.values, aspect="auto", cmap="cividis",
            vmin=max(0.4, vals_clean.min() - 0.05),
            vmax=min(1.0, vals_clean.max() + 0.05),
        )
        plt.colorbar(im, ax=ax, label="Mean AUC")
        ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
        ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index)
        for (i, j), val in np.ndenumerate(pivot.values):
            if not np.isnan(val):
                ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                        fontsize=10, color="black")
        ax.set_xlabel("Activity Weight")
        ax.set_ylabel("Transition Aggressiveness")
        ax.set_title("Mean ROC-AUC: Transition vs Activity Weight\n(anomaly_rate=0)")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "auc_heatmap.png", dpi=150)
        plt.close()

    # ── Plot 2: Real lead-time vs transition ──────────────────────────────
    r_lt = results[results["real_lead_time_min"].notna()]
    if not r_lt.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        _lt_styles = {"conservative": ("-", "o"), "moderate": ("--", "s"), "aggressive": (":", "^")}
        for tn in TRANSITION_LEVELS:
            sub = r_lt[r_lt["transition"] == tn]
            grp = sub.groupby("anomaly_rate")["real_lead_time_min"].mean()
            ls, mk = _lt_styles.get(tn, ("-", "o"))
            ax.plot(grp.index, grp.values, marker=mk, linestyle=ls, lw=2, markersize=7, label=f"trans={tn}")
        ax.set_xlabel("Anomaly Injection Rate", fontsize=11)
        ax.set_ylabel("Mean Real Lead Time (minutes)", fontsize=11)
        ax.set_title("Real Vasopressor Lead-Time vs Anomaly Injection Rate", fontsize=12)
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "real_lead_time_vs_anomaly.png", dpi=150)
        plt.close()

    # ── Plot 3: Sensitivity vs False Alarm (anomaly_rate=0.10) ───────────
    r_sens = results[
        (results["anomaly_rate"] == 0.10) &
        results["sensitivity"].notna() &
        results["false_alarm_rate"].notna()
    ]
    if not r_sens.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        markers = {"conservative": "o", "moderate": "s", "aggressive": "^"}
        colors  = {"low": "steelblue", "medium": "orange", "high": "crimson"}
        for tn in TRANSITION_LEVELS:
            for an in ACTIVITY_WEIGHT_LEVELS:
                sub = r_sens[
                    (r_sens["transition"] == tn) &
                    (r_sens["activity_wt"] == an)
                ]
                if sub.empty: continue
                ax.scatter(
                    sub["false_alarm_rate"], sub["sensitivity"],
                    marker=markers.get(tn, "o"),
                    color=colors.get(an, "grey"),
                    s=80, alpha=0.8,
                    label=f"{tn}/{an}",
                )
        ax.set_xlabel("False Alarm Rate",  fontsize=11)
        ax.set_ylabel("Sensitivity",       fontsize=11)
        ax.set_title("Sensitivity vs False Alarm Rate\n(anomaly_rate=0.10)", fontsize=12)
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles[:9], labels[:9], fontsize=8, loc="lower right")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "sensitivity_vs_false_alarm.png", dpi=150)
        plt.close()

    # ── Plot 4: Alert rate by threshold ──────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 5))
    _ar_styles = {"conservative": ("-", "o"), "moderate": ("--", "s"), "aggressive": (":", "^")}
    for tn in TRANSITION_LEVELS:
        sub = results[results["transition"] == tn]
        grp = sub.groupby("threshold")["alert_rate"].mean()
        ls, mk = _ar_styles.get(tn, ("-", "s"))
        ax.plot(grp.index, grp.values, marker=mk, linestyle=ls, lw=2, markersize=7, label=f"trans={tn}")
    ax.set_xlabel("Alert Threshold",  fontsize=11)
    ax.set_ylabel("Mean Alert Rate",  fontsize=11)
    ax.set_title("Alert Rate vs Threshold by Transition Setting", fontsize=12)
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "alert_rate_vs_threshold.png", dpi=150)
    plt.close()

    # ── Plot 5: AUC with vs without behavioral context ───────────────────
    r_beh = results[
        results["auc_with_behavioral"].notna() &
        results["auc_icu_only"].notna()
    ]
    if not r_beh.empty:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(r_beh["auc_icu_only"], r_beh["auc_with_behavioral"],
                   alpha=0.6, color="steelblue", s=40, marker="D")
        lim = [
            min(r_beh[["auc_icu_only", "auc_with_behavioral"]].min()),
            max(r_beh[["auc_icu_only", "auc_with_behavioral"]].max()),
        ]
        ax.plot(lim, lim, "k--", lw=1, label="No difference")
        ax.set_xlabel("AUC — ICU vitals only",         fontsize=11)
        ax.set_ylabel("AUC — with behavioral context", fontsize=11)
        ax.set_title(
            "Impact of Behavioral Context on AUC\n"
            "(above diagonal = behavioral helps)",
            fontsize=11
        )
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "behavioral_context_impact.png", dpi=150)
        plt.close()

    # ── Plot 6: AUC vs anomaly rate by transition ─────────────────────────
    fig, ax = plt.subplots(figsize=(8, 5))
    _auc_styles = {"conservative": ("-", "o"), "moderate": ("--", "s"), "aggressive": (":", "^")}
    for tn in TRANSITION_LEVELS:
        sub = r_valid[r_valid["transition"] == tn]
        grp = sub.groupby("anomaly_rate")["auc"].mean()
        ls, mk = _auc_styles.get(tn, ("-", "o"))
        ax.plot(grp.index, grp.values, marker=mk, linestyle=ls, lw=2, markersize=7, label=f"trans={tn}")
    ax.set_xlabel("Anomaly Injection Rate", fontsize=11)
    ax.set_ylabel("Mean AUC",               fontsize=11)
    ax.set_title("AUC Robustness vs Anomaly Injection Rate", fontsize=12)
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "auc_vs_anomaly_rate.png", dpi=150)
    plt.close()

    print("\nExperiment output files:")
    for f in ["experiment_results.csv", "auc_heatmap.png",
              "real_lead_time_vs_anomaly.png",
              "sensitivity_vs_false_alarm.png",
              "alert_rate_vs_threshold.png",
              "behavioral_context_impact.png",
              "auc_vs_anomaly_rate.png"]:
        print(f"  {f}")


# ============================================================
# MAIN
# ============================================================

def run_experiments():
    print("\n========== PEMDT v2 Experiments ==========\n")

    if not Path(DASHBOARD_FILE).exists():
        raise FileNotFoundError(
            f"{DASHBOARD_FILE} not found. Run pemdt_pipeline.py first."
        )

    fused_df = pd.read_csv(DASHBOARD_FILE, low_memory=False)
    fused_df["timestamp"] = pd.to_datetime(fused_df["timestamp"], errors="coerce")

    if "first_vasopressor_time" in fused_df.columns:
        fused_df["first_vasopressor_time"] = pd.to_datetime(
            fused_df["first_vasopressor_time"], errors="coerce"
        )

    fused_df["is_deceased"] = fused_df["is_deceased"].fillna(0).astype(int)

    print(f"Dashboard: {fused_df.shape}")
    print(f"Patients : {fused_df['patient_id'].nunique()}")
    print(f"Deaths   : {fused_df.groupby('patient_id')['is_deceased'].max().sum()}")
    if "has_behavioral_data" in fused_df.columns:
        n_beh = fused_df[fused_df["has_behavioral_data"] == True]["patient_id"].nunique()
        print(f"With CASAS behavioral data: {n_beh} patients")

    # ── Stratified sample for speed ───────────────────────────────────────
    print("\nSampling cohort for experiments...")
    deceased    = fused_df[fused_df["is_deceased"] == 1]["patient_id"].unique()
    alive       = fused_df[fused_df["is_deceased"] == 0]["patient_id"].unique()
    rng_sample  = np.random.default_rng(RANDOM_SEED)
    sample_pids = np.concatenate([
        rng_sample.choice(deceased, min(2500, len(deceased)), replace=False),
        rng_sample.choice(alive,    min(5000, len(alive)),    replace=False),
    ])
    fused_df = fused_df[fused_df["patient_id"].isin(sample_pids)].copy()
    print(f"Experiment cohort: {fused_df['patient_id'].nunique()} patients "
          f"({fused_df[fused_df['is_deceased']==1]['patient_id'].nunique()} deceased)")

    results = run_all_experiments(fused_df)
    analyse_experiments(results)

    print(f"\n✅ All done — outputs in: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    run_experiments()


========== PEMDT v2 Experiments ==========

Dashboard: (9890935, 36)
Patients : 64691
Deaths   : 21267
With CASAS behavioral data: 80 patients

Sampling cohort for experiments...
Experiment cohort: 7500 patients (2500 deceased)

Total experiment runs: 180  (3 trans × 3 act × 4 thresh × 5 anomaly)
  Run   1/180 — trans=conservative  act=low     thresh=0.3  anomaly=0.0
  Run  30/180 — trans=conservative  act=medium  thresh=0.4  anomaly=0.2
  Run  60/180 — trans=conservative  act=high    thresh=0.6  anomaly=0.2
  Run  90/180 — trans=moderate      act=medium  thresh=0.4  anomaly=0.2
  Run 120/180 — trans=moderate      act=high    thresh=0.6  anomaly=0.2
  Run 150/180 — trans=aggressive    act=medium  thresh=0.4  anomaly=0.2
  Run 180/180 — trans=aggressive    act=high    thresh=0.6  anomaly=0.2

✅ All 180 runs complete.

=== Experiment Analysis ===

 Best config by AUC:
   transition: conservative
   activity_wt: low
   threshold: 0.3
   anomaly_rate: 0.1
   auc: 0.6947
   auc_ci_lo: 0.68

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

OUTPUT_DIR = Path(".")

# ── Model parameters (must match your PEMDT config) ───────────────────────
GLOBAL_STATS = {
    "SBP":        (120, 20),
    "HRV_proxy":  (0.15, 0.10),
    "SpO2":       (97,   3),
    "Resp_rate":  (18,   5),
    "MAP":        (80,   15),
    "Lactate":    (2.0,  1.5),
    "WBC":        (10,   5),
    "Creatinine": (1.2,  0.8),
}

FEATURE_WEIGHTS = {
    "SBP":        1.3,
    "HRV_proxy":  1.5,
    "SpO2":       1.4,
    "Resp_rate":  1.2,
    "MAP":        1.4,
    "Lactate":    1.6,
    "WBC":        1.0,
    "Creatinine": 1.1,
}

UNSTABLE_Z = {
    "SBP":        -2.0,
    "HRV_proxy":  -2.0,
    "SpO2":       -2.0,
    "Resp_rate":  +2.0,
    "MAP":        -2.0,
    "Lactate":    +2.0,
    "WBC":        +2.0,
    "Creatinine": +2.0,
}

# Display labels — order controls plot order
DISPLAY_ORDER = [
    "SpO2", "Resp_rate", "MAP", "Lactate",
    "Creatinine", "SBP", "WBC", "HRV_proxy"
]
DISPLAY_LABELS = {
    "SpO2":       "Oxygen Saturation",
    "Resp_rate":  "Respiratory Rate",
    "MAP":        "Mean Arterial Pressure",
    "Lactate":    "Lactate",
    "Creatinine": "Creatinine",
    "SBP":        "Systolic BP",
    "WBC":        "White Cell Count",
    "HRV_proxy":  "HRV (variability)",
}
SHORT_LABELS = {
    "SpO2":       "SpO₂",
    "Resp_rate":  "Resp Rate",
    "MAP":        "MAP",
    "Lactate":    "Lactate",
    "Creatinine": "Creatinine",
    "SBP":        "SBP",
    "WBC":        "WBC",
    "HRV_proxy":  "HRV",
}
IS_LAB = {
    "SpO2": False, "Resp_rate": False, "MAP": False,
    "Lactate": True, "Creatinine": True, "SBP": False,
    "WBC": True, "HRV_proxy": False,
}


# ── STEP 1: COMPUTE FREQUENCIES FROM DATA ────────────────────────────────

def compute_frequencies(dashboard_path="PEMDT_dashboard_cohort.csv"):
    """
    Compute LLR and delta-f dominant feature frequencies from raw data.
    Saves results to explainability_frequencies.csv for reproducibility.
    Returns two dicts: llr_freq, delta_freq (feature -> percentage)
    """
    print("Loading dashboard...")
    dashboard = pd.read_csv(dashboard_path, low_memory=False)

    dashboard = dashboard.sort_values(["patient_id", "timestamp"]).reset_index(drop=True)

    features  = [f for f in DISPLAY_ORDER if f in dashboard.columns]

    print(f"  {len(dashboard):,} windows, {len(features)} features found")

    # ── LLR per feature ──────────────────────────────────────────────────
    print("Computing LLR dominant features...")
    llr_scores = {}
    for feat in features:
        mu, sigma = GLOBAL_STATS[feat]
        w         = FEATURE_WEIGHTS[feat]
        z_star    = UNSTABLE_Z[feat]
        z         = (dashboard[feat] - mu) / sigma
        log_unstable = -0.5 * ((z - z_star) / 1.5) ** 2
        log_stable   = -0.5 * (z ** 2)
        llr_scores[feat] = w * (log_unstable - log_stable)

    llr_df       = pd.DataFrame(llr_scores)
    dominant_llr = llr_df.abs().idxmax(axis=1)
    llr_freq     = (dominant_llr.value_counts(normalize=True) * 100).round(2)

    # ── Delta-f per feature ───────────────────────────────────────────────
    print("Computing delta-f dominant features...")
    delta_scores = {}
    for feat in features:
        sigma = GLOBAL_STATS[feat][1]
        prev  = dashboard.groupby("patient_id")[feat].shift(1)
        delta_scores[feat] = ((dashboard[feat] - prev).abs() / sigma).fillna(0)

    delta_df       = pd.DataFrame(delta_scores)
    dominant_delta = delta_df.abs().idxmax(axis=1)
    delta_freq     = (dominant_delta.value_counts(normalize=True) * 100).round(2)

    # ── Agreement ────────────────────────────────────────────────────────
    agreement_pct = (dominant_llr == dominant_delta).mean() * 100

    # ── Save to CSV ───────────────────────────────────────────────────────
    rows = []
    for feat in DISPLAY_ORDER:
        rows.append({
            "feature":     feat,
            "llr_pct":     llr_freq.get(feat, 0.0),
            "delta_pct":   delta_freq.get(feat, 0.0),
            "difference":  round(llr_freq.get(feat, 0.0) - delta_freq.get(feat, 0.0), 2),
            "is_lab":      IS_LAB[feat],
        })
    freq_df = pd.DataFrame(rows)
    freq_df.to_csv(OUTPUT_DIR / "explainability_frequencies.csv", index=False)
    print(f"\n✅ Frequencies saved to explainability_frequencies.csv")
    print(f"   Agreement: {agreement_pct:.1f}%")
    print(freq_df.to_string(index=False))

    return freq_df, agreement_pct


def load_frequencies():
    """
    Load pre-computed frequencies from CSV.
    Use this to regenerate plots without reloading the full dashboard.
    """
    path = OUTPUT_DIR / "explainability_frequencies.csv"
    if not path.exists():
        raise FileNotFoundError(
            "explainability_frequencies.csv not found. "
            "Run compute_frequencies() first."
        )
    return pd.read_csv(path)


# ── STEP 2: GENERATE PLOTS ────────────────────────────────────────────────

def plot_all(freq_df, agreement_pct):
    """Generate all three explainability plots from a frequencies dataframe."""

    llr_pct   = freq_df["llr_pct"].tolist()
    delta_pct = freq_df["delta_pct"].tolist()
    is_lab    = freq_df["is_lab"].tolist()
    features  = [DISPLAY_LABELS[f] for f in freq_df["feature"]]
    short     = [SHORT_LABELS[f]   for f in freq_df["feature"]]

    # ── Plot 1: Grouped bar ───────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 6))

    x     = np.arange(len(features))
    width = 0.35

    llr_colors   = ["#1ABC9C" if lab else "#2C6FAC" for lab in is_lab]
    delta_colors = ["#E67E22" if lab else "#C0392B" for lab in is_lab]

    bars1 = ax.bar(x - width/2, llr_pct,   width,
                   color=llr_colors,   edgecolor="black", linewidth=0.8, hatch="//")
    bars2 = ax.bar(x + width/2, delta_pct, width,
                   color=delta_colors, edgecolor="black", linewidth=0.8, hatch="xx")

    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                f"{h:.1f}%", ha="center", va="bottom", fontsize=8.5)
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                f"{h:.1f}%", ha="center", va="bottom", fontsize=8.5)

    ax.set_xticks(x)
    ax.set_xticklabels(features, rotation=20, ha="right", fontsize=10)
    ax.set_ylabel("% of Windows as Dominant Feature", fontsize=11)
    ax.set_title("Feature Attribution Frequency: LLR (Λf) vs Change-Based (δf)\n"
                 "Teal/orange bars = laboratory biomarkers", fontsize=12)
    ax.set_ylim(0, max(max(llr_pct), max(delta_pct)) * 1.2)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.set_facecolor("#FAFAFA")

    # Annotation — point to lab bars with lowest delta value
    min_delta_lab_idx = min(
        [i for i, lab in enumerate(is_lab) if lab],
        key=lambda i: delta_pct[i]
    )
    ax.annotate("Lab biomarkers nearly\ninvisible to δf",
                xy=(min_delta_lab_idx + width/2, delta_pct[min_delta_lab_idx] + 0.3),
                xytext=(min_delta_lab_idx + 1.5, max(delta_pct) * 0.6),
                arrowprops=dict(arrowstyle="->", color="#E67E22", lw=1.5),
                fontsize=9, color="#E67E22",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                          edgecolor="#E67E22", alpha=0.8))

    ax.legend(handles=[
        mpatches.Patch(facecolor="#2C6FAC", hatch="//", edgecolor="black", label="Vitals — LLR (Λf)"),
        mpatches.Patch(facecolor="#C0392B", hatch="xx", edgecolor="black", label="Vitals — δf"),
        mpatches.Patch(facecolor="#1ABC9C", hatch="//", edgecolor="black", label="Labs — LLR (Λf)"),
        mpatches.Patch(facecolor="#E67E22", hatch="xx", edgecolor="black", label="Labs — δf"),
    ], fontsize=9, loc="upper right")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "explainability_plot1_frequency.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("✅ Saved: explainability_plot1_frequency.png")

    # ── Plot 2: Diverging bar ─────────────────────────────────────────────
    diffs  = [l - d for l, d in zip(llr_pct, delta_pct)]
    colors = []
    for i, d in enumerate(diffs):
        if is_lab[i]:
            colors.append("#1ABC9C" if d > 0 else "#E67E22")
        else:
            colors.append("#2C6FAC" if d > 0 else "#C0392B")

    fig, ax = plt.subplots(figsize=(9, 6))
    div_hatches = ["//" if d > 0 else "xx" for d in diffs]
    bars = ax.barh(short, diffs, color=colors,
                   edgecolor="black", linewidth=0.8, height=0.6)
    for bar, hatch in zip(bars, div_hatches):
        bar.set_hatch(hatch)
    ax.axvline(0, color="black", linewidth=1.2)

    for bar, val in zip(bars, diffs):
        ax.text(val + (0.3 if val >= 0 else -0.3),
                bar.get_y() + bar.get_height()/2,
                f"{val:+.1f}%", va="center",
                ha="left" if val >= 0 else "right", fontsize=10)

    ax.set_xlabel("LLR attribution % − δf attribution %", fontsize=11)
    ax.set_title("Attribution Difference per Feature  (LLR − δf)\n"
                 "Positive = LLR attributes more  |  Negative = δf attributes more",
                 fontsize=11)
    ax.grid(axis="x", alpha=0.3, linestyle="--")
    ax.set_facecolor("#FAFAFA")
    ax.legend(handles=[
        mpatches.Patch(facecolor="#2C6FAC", hatch="//", edgecolor="black", label="Vitals: LLR > δf"),
        mpatches.Patch(facecolor="#C0392B", hatch="xx", edgecolor="black", label="Vitals: δf > LLR"),
        mpatches.Patch(facecolor="#1ABC9C", hatch="//", edgecolor="black", label="Labs: LLR > δf"),
        mpatches.Patch(facecolor="#E67E22", hatch="xx", edgecolor="black", label="Labs: δf > LLR"),
    ], fontsize=9, loc="lower right")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "explainability_plot2_difference.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("✅ Saved: explainability_plot2_difference.png")

    # ── Plot 3: Pie ───────────────────────────────────────────────────────
    total    = 9_826_244
    agree_n  = int(total * agreement_pct / 100)

    lab_feats      = [f for f in freq_df["feature"] if IS_LAB[f]]
    lab_llr_total  = freq_df[freq_df["feature"].isin(lab_feats)]["llr_pct"].sum()
    lab_delt_total = freq_df[freq_df["feature"].isin(lab_feats)]["delta_pct"].sum()
    lab_disagree_n = int((lab_llr_total - lab_delt_total) / 100 * total)
    other_n        = total - agree_n - lab_disagree_n

    sizes  = [agree_n, lab_disagree_n, other_n]
    labels = [
        f"Agreement\n{agreement_pct:.1f}%\n({agree_n:,} windows)",
        f"Disagree — Lab features\n(LLR >> δf)\n{lab_disagree_n/total*100:.1f}%",
        f"Disagree — Vital signs\n(methods differ)\n{other_n/total*100:.1f}%",
    ]

    fig, ax = plt.subplots(figsize=(7, 7))
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels,
        colors=["#27AE60", "#1ABC9C", "#C0392B"],
        explode=[0, 0.07, 0],
        autopct="%1.1f%%", startangle=90, pctdistance=0.78,
        textprops={"fontsize": 9.5},
        wedgeprops={"edgecolor": "black", "linewidth": 2}
    )
    pie_hatches = ["//", "..", "xx"]
    for wedge, hatch in zip(wedges, pie_hatches):
        wedge.set_hatch(hatch)
    for at in autotexts:
        at.set_fontsize(9.5)
        at.set_fontweight("bold")

    ax.set_title(f"LLR vs δf Feature Agreement\nAcross {total:,} windows  "
                 f"({100-agreement_pct:.1f}% disagreement overall)", fontsize=11)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "explainability_plot3_agreement.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("✅ Saved: explainability_plot3_agreement.png")


# ── Initially compute from data (first run or after data changes) ─────
# freq_df, agreement_pct = compute_frequencies("PEMDT_dashboard_cohort.csv")

# # ── Then load from saved CSV (fast rerun for plot tweaks only) ───
freq_df        = load_frequencies()
agreement_pct  = 20.3  # paste value from your compute run

plot_all(freq_df, agreement_pct)
print("\n✅ All explainability plots generated.")

✅ Saved: explainability_plot1_frequency.png
✅ Saved: explainability_plot2_difference.png
✅ Saved: explainability_plot3_agreement.png

✅ All explainability plots generated.
